# BGE Human-FT + CatBoost residual bank — выполненный Kaggle-прогон\n\nНоутбук объединяет историю экспериментов с selective E5+BGE, категорийными весами и LLM-soft CatBoost bank, а финальная ячейка проверяет residual-ансамбль поверх нового BGE Human-FT на 2×T4.\n\n**Финальный результат:** BGE Human-FT имеет public LB **0.543256**, manual Macro PR-AUC **0.820226** и LLM holdout **0.803502**. Три LLM-teacher CatBoost и human Meta-CatBoost (`d6/d7/d8`, component-safe 5-fold) не дали прироста: лучший выбранный вес `alpha=0`; уже при `alpha=0.025` tune ухудшается на `−0.000071`.\n\n**Вывод:** CatBoost residual поверх BGE Human-FT не использовать в submission; текущий лучший вариант — чистый BGE Human-FT. Human-метрика не полностью untouched, поскольку Human-FT мог видеть human labels. Время финального прогона — **1.06 часа**.

In [1]:
import os

for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        if filename.endswith(".joblib"):
            print(os.path.join(dirname, filename))

/kaggle/input/models/danilzhukovv/ecup-baseline-logreg-l12/scikitlearn/default/1/baseline_logreg_l12.joblib


In [4]:
import json, gc
from pathlib import Path
import numpy as np, pandas as pd, polars as pl
from sklearn.metrics import average_precision_score

BASE=Path("/kaggle/input/datasets/mihailivanovvvv/hakaton-ozon-math-items")
HUMAN,HI=BASE/"matches.parquet",BASE/"items_human.parquet"
LLM,LI=BASE/"matches_llm.parquet",BASE/"items.parquet"
OUT=Path("/kaggle/working/selective_bge"); OUT.mkdir(exist_ok=True)

def locate(name):
    for root in (Path("/kaggle/working"),Path("/kaggle/input")):
        p=list(root.rglob(name))
        if p:return p[0]
    raise FileNotFoundError(name)

BP=np.load(locate("bge_manual.npy")); EP=np.load(locate("e5_manual_val.npy"))
BL=np.load(locate("bge_llm.npy")); EL=np.load(locate("e5_llm.npy"))
print("arrays:",len(BP),len(EP),len(BL),len(EL))

def component_mask(a,b,frac,seed):
    parent={}
    def find(x):
        parent.setdefault(x,x)
        while parent[x]!=x:
            parent[x]=parent[parent[x]]; x=parent[x]
        return x
    for x,y in zip(a,b):
        x,y=find(x),find(y)
        if x!=y:parent[y]=x
    comp=np.fromiter((find(x) for x in a),np.int64,len(a))
    u=np.unique(comp); rng=np.random.RandomState(seed)
    chosen=set(u[rng.rand(len(u))<frac])
    return np.fromiter((x in chosen for x in comp),bool,len(comp))

def add_category(df,items):
    need=df.select(pl.col("id1").alias("id")).unique()
    cats=(pl.scan_parquet(items).select("id","category")
          .join(need.lazy(),on="id",how="semi").collect(engine="streaming"))
    z=(df.with_row_index("_row").join(cats.rename({"id":"id1"}),on="id1",how="left")
       .sort("_row").drop("_row"))
    assert z["category"].null_count()==0
    return z

# Тот же untouched manual split: 72 948 пар
hm=pl.read_parquet(HUMAN,columns=["id1","id2","target"])
um=component_mask(hm["id1"].to_numpy(),hm["id2"].to_numpy(),.20,42)
hv=add_category(hm.filter(pl.Series(um)),HI)
assert len(hv)==len(EP), (len(hv),len(EP))
BM=BP[um]; y=hv["target"].to_numpy().astype(np.int8)
cat=hv["category"].cast(pl.String).to_numpy()

# Независимые component-safe tune/eval половины
tune=component_mask(hv["id1"].to_numpy(),hv["id2"].to_numpy(),.50,2026)
evaluation=~tune

def rank(x):return pd.Series(x).rank(method="average",pct=True).to_numpy()

def selective(e5,bge,categories,top_frac,bge_weight):
    out=np.empty(len(e5),np.float64)
    for c in np.unique(categories):
        ix=np.flatnonzero(categories==c); n=len(ix)
        base=rank(e5[ix]); out[ix]=base
        k=max(1,int(np.ceil(n*top_frac)))
        local=np.argsort(e5[ix],kind="stable")[-k:]
        chosen=ix[local]
        mix=(1-bge_weight)*rank(e5[chosen])+bge_weight*rank(bge[chosen])
        out[chosen]=(n-k)/n+rank(mix)*k/n
    return out

def macro(y,p,c,mask=None):
    if mask is None:mask=np.ones(len(y),bool)
    scores=[]
    for k in np.unique(c[mask]):
        m=mask&(c==k)
        if len(np.unique(y[m]))>1:scores.append(average_precision_score(y[m],p[m]))
    return float(np.mean(scores))

base={"tune":macro(y,EP,cat,tune),"eval":macro(y,EP,cat,evaluation),
      "full":macro(y,EP,cat)}
rows=[]
for q in (.05,.10,.15,.20,.25,.30):
    for w in (0,.025,.05,.075,.10,.125,.15,.20):
        p=selective(EP,BM,cat,q,w)
        rows.append({"top_frac":q,"bge_weight":w,
                     "tune":macro(y,p,cat,tune),
                     "eval":macro(y,p,cat,evaluation),
                     "full":macro(y,p,cat)})
grid=pd.DataFrame(rows).sort_values("tune",ascending=False)
best=grid.iloc[0].to_dict()
print("\nE5 baseline:",base)
print("\nTop-10 по tune:\n",grid.head(10).to_string(index=False))
print("\nSelected:",best)

# Точный LLM holdout 191 555
lm=pl.read_parquet(LLM,columns=["id1","id2","target"])
ul=component_mask(lm["id1"].to_numpy(),lm["id2"].to_numpy(),.03,13)
lv=(lm.filter(pl.Series(ul))
    .filter((pl.col("target")<=.2)|(pl.col("target")>=.8))
    .with_columns((pl.col("target")>=.5).cast(pl.Int8).alias("target")))
lv=add_category(lv,LI)
assert len(lv)==len(EL)==191555
yl=lv["target"].to_numpy(); cl=lv["category"].cast(pl.String).to_numpy()
lp=selective(EL,BL,cl,best["top_frac"],best["bge_weight"])
llm={"e5":macro(yl,EL,cl),"selective":macro(yl,lp,cl)}
accepted=(best["eval"]>base["eval"]+.0005 and best["full"]>base["full"])

result={"selected_top_frac":best["top_frac"],"selected_bge_weight":best["bge_weight"],
        "manual_e5":base,"manual_selective":{k:best[k] for k in ("tune","eval","full")},
        "llm":llm,"recommended":bool(accepted)}
grid.to_csv(OUT/"grid.csv",index=False)
(OUT/"selective_config.json").write_text(json.dumps(result,ensure_ascii=False,indent=2))
print("\nFINAL:\n",json.dumps(result,ensure_ascii=False,indent=2))
print("\nВердикт:","СОБИРАТЬ selective-контейнер" if accepted else "ОСТАВИТЬ E5 SOLO")

arrays: 365654 72948 191555 191555

E5 baseline: {'tune': 0.774751154830206, 'eval': 0.7667150912382683, 'full': 0.7701124939243256}

Top-10 по tune:
  top_frac  bge_weight     tune     eval     full
     0.30       0.150 0.775613 0.769215 0.771864
     0.25       0.150 0.775600 0.769180 0.771761
     0.30       0.125 0.775596 0.769049 0.771683
     0.30       0.075 0.775575 0.768545 0.771444
     0.25       0.200 0.775568 0.769343 0.771919
     0.20       0.200 0.775566 0.769234 0.771834
     0.30       0.200 0.775554 0.769303 0.771898
     0.25       0.075 0.775542 0.768336 0.771323
     0.25       0.125 0.775489 0.769030 0.771587
     0.20       0.150 0.775479 0.769095 0.771610

Selected: {'top_frac': 0.3, 'bge_weight': 0.15, 'tune': 0.7756132139294939, 'eval': 0.7692149738035828, 'full': 0.7718637349016714}

FINAL:
 {
  "selected_top_frac": 0.3,
  "selected_bge_weight": 0.15,
  "manual_e5": {
    "tune": 0.774751154830206,
    "eval": 0.7667150912382683,
    "full": 0.7701124939243

In [7]:
import contextlib, gc, json, os, re, shutil, time
from difflib import SequenceMatcher
from pathlib import Path

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "true"

import numpy as np
import pandas as pd
import polars as pl
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
import torch
import torch.nn as nn
from sklearn.metrics import average_precision_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer, PreTrainedTokenizerFast

ROOT = Path("/kaggle/input")
BASE = Path("/kaggle/input/datasets/mihailivanovvvv/hakaton-ozon-math-items")
MATCHES, ITEMS = BASE / "matches.parquet", BASE / "items_human.parquet"
OUT = Path("/kaggle/working/v53_category_weights"); OUT.mkdir(exist_ok=True)
TOP_FRAC = .10
WEIGHT_GRID = tuple(np.arange(0, .51, .05).round(2))
DEV = torch.device("cuda:0")
assert torch.cuda.device_count() == 2, "Включи 2x T4"


def find_model(kind):
    if kind == "e5":
        hits = list(ROOT.rglob("hybrid_heads.pt"))
        hits.sort(key=lambda p: ("v5-3" not in str(p).lower() and "v5_3" not in str(p).lower(), len(str(p))))
        if hits:
            return hits[0].parent
    else:
        hits = []
        for p in ROOT.rglob("config.json"):
            try:
                c = json.loads(p.read_text())
                if (c.get("vocab_size") == 46166 and c.get("hidden_size") == 1024
                        and c.get("num_hidden_layers") == 24
                        and (p.parent / "model.safetensors").exists()):
                    hits.append(p.parent)
            except Exception:
                pass
        if hits:
            return sorted(hits, key=lambda p: len(str(p)))[0]
    raise FileNotFoundError(f"Не найдена модель {kind} в /kaggle/input")


E5_DIR, BGE_DIR = find_model("e5"), find_model("bge")
print("GPU:", [torch.cuda.get_device_name(i) for i in range(2)])
print("E5 V5.3:", E5_DIR, "\nBGE:", BGE_DIR)


def component_mask(a, b, frac, seed):
    parent = {}
    def find(x):
        parent.setdefault(x, x)
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    for x, y in zip(a, b):
        x, y = find(x), find(y)
        if x != y:
            parent[y] = x
    comp = np.fromiter((find(x) for x in a), np.int64, len(a))
    groups = np.unique(comp); rng = np.random.RandomState(seed)
    chosen = set(groups[rng.rand(len(groups)) < frac])
    return np.fromiter((x in chosen for x in comp), bool, len(comp))


# Exact V5.3 preprocessing
SPACE_RE = re.compile(r"\s+")
MULTIPLY_RE = re.compile(r"[×хХ*]")
TOKEN_RE = re.compile(r"[0-9a-zа-я]+", re.I)
SIZE_RE = re.compile(r"(?:размер|р-р|size)\s*[:=\-]?\s*([0-9]{2,3}(?:[./-][0-9]{1,3})?|xxxs|xxs|xs|s|m|l|xl|xxl|xxxl)", re.I)
UNIT_RE = re.compile(r"(\d+(?:[.,]\d+)?)\s*(мл|ml|л|l|мг|mg|г|gr|g|кг|kg|мм|mm|см|cm|м|m|вт|w|квт|kw|шт|pcs|pack)\b", re.I)
PACK_RE = re.compile(r"(\d+)\s*[xх×*]\s*(\d+(?:[.,]\d+)?)\s*(мл|ml|л|l|мг|mg|г|gr|g|кг|kg|шт|pcs)", re.I)
KEY_ORDER = ["бренд","brand","производитель","артикул","партномер","part number","partnumber","oem","код","sku","модель","model","размер","size","рост","обхват","пол","gender","цвет","color","материал","material","объем","обьем","volume","вес","weight","количество","комплектация","упаков","тип","type"]
ALIASES = {
    "brand":["бренд","brand","производитель"],
    "article":["артикул","sku","oem","партномер","part number","partnumber","код производителя"],
    "model":["модель","model"],
    "size":["размер производителя","размер обуви","размер одежды","размер","size","рост"],
    "color":["основной цвет","цвет","color","расцветка"],
    "material":["материал верха","материал","material"],
    "gender":["пол","gender"],
}
HOMO = str.maketrans({"а":"a","в":"b","с":"c","е":"e","н":"h","к":"k","м":"m","о":"o","р":"p","т":"t","х":"x","у":"y"})
COLORS = {x.replace("ё","е"): y for x,y in {
    "черный":"черный","чёрный":"черный","белый":"белый","серый":"серый","серебристый":"серебристый",
    "красный":"красный","бордовый":"бордовый","синий":"синий","голубой":"голубой","зеленый":"зеленый",
    "зелёный":"зеленый","желтый":"желтый","жёлтый":"желтый","оранжевый":"оранжевый","розовый":"розовый",
    "фиолетовый":"фиолетовый","бежевый":"бежевый","коричневый":"коричневый","золотой":"золотой","золотистый":"золотой"
}.items()}


def norm(x):
    if x is None: return ""
    return SPACE_RE.sub(" ", MULTIPLY_RE.sub("x", str(x).lower().replace("ё", "е").replace(",", "."))).strip()


def attrs_dict(x):
    if isinstance(x, dict): d = x
    elif isinstance(x, str):
        try: d = json.loads(x)
        except Exception: d = {}
    else: d = {}
    return {norm(k):norm(v) for k,v in d.items() if norm(k) and norm(v)} if isinstance(d, dict) else {}


def pick(d, aliases):
    for alias in aliases:
        for k,v in d.items():
            if alias in k: return v
    return ""


def compact(x, limit=96):
    return SPACE_RE.sub(" ", re.sub(r"[^0-9a-zа-я.+/_\- ]+", " ", norm(x))).strip()[:limit]


def code_norm(x): return re.sub(r"[^0-9a-z]+", "", compact(x).translate(HOMO))
def size_norm(x): return re.sub(r"\s*/\s*", "/", re.sub(r"\s*-\s*", "-", compact(x,64).replace("–","-").replace("—","-")))


def color_norm(x):
    text = compact(x, 64); found = [v for k,v in COLORS.items() if k in text]
    return "/".join(sorted(set(found))) if found else text


def fallback_color(name):
    text=norm(name); found=[v for k,v in COLORS.items() if k in text]
    return "/".join(sorted(set(found)))


def quantity_tokens(text):
    def convert(v,u):
        u=u.lower()
        if u in {"мл","ml"}: return "volume_ml",v
        if u in {"л","l"}: return "volume_ml",v*1000
        if u in {"мг","mg"}: return "mass_g",v/1000
        if u in {"г","gr","g"}: return "mass_g",v
        if u in {"кг","kg"}: return "mass_g",v*1000
        if u in {"мм","mm"}: return "length_mm",v
        if u in {"см","cm"}: return "length_mm",v*10
        if u in {"м","m"}: return "length_mm",v*1000
        if u in {"вт","w"}: return "power_w",v
        if u in {"квт","kw"}: return "power_w",v*1000
        return "count",v
    text=norm(text); out=set()
    for c,a,u in PACK_RE.findall(text):
        kind,base=convert(float(a.replace(",",".")),u); count=float(c)
        out|={f"{kind}:{round(count*base,4)}",f"pack_count:{round(count,4)}"}
    for a,u in UNIT_RE.findall(text):
        kind,base=convert(float(a.replace(",",".")),u); out.add(f"{kind}:{round(base,4)}")
    return frozenset(out)


class Item:
    __slots__=("text","name","tokens","brand","article","model","size","color","material","gender","quantity","category")
    def __init__(self,name,attributes,category):
        d=attrs_dict(attributes); self.name=norm(name); self.tokens=frozenset(x for x in TOKEN_RE.findall(self.name) if len(x)>=2)
        self.brand=compact(pick(d,ALIASES["brand"])); self.article=code_norm(pick(d,ALIASES["article"])); self.model=code_norm(pick(d,ALIASES["model"]))
        m=SIZE_RE.search(self.name); self.size=size_norm(pick(d,ALIASES["size"])) or (size_norm(m.group(1)) if m else "")
        self.color=color_norm(pick(d,ALIASES["color"])) or fallback_color(self.name); self.material=compact(pick(d,ALIASES["material"])); self.gender=compact(pick(d,ALIASES["gender"])); self.category=str(category)
        used=set(); priority=[]
        for wanted in KEY_ORDER:
            for k,v in d.items():
                if k not in used and wanted in k: priority.append(f"{k}: {v}"); used.add(k)
        attrs=" ; ".join(priority+[f"{k}: {v}" for k,v in d.items() if k not in used])[:620]
        self.text=f"категория: {norm(category)} | название: {self.name} | атрибуты: {attrs}"
        self.quantity=quantity_tokens(self.name+" "+" ".join(d.values())[:1800])


def jaccard(a,b): return len(a&b)/len(a|b) if a|b else 0.
def contain(a,b): return len(a&b)/max(1,min(len(a),len(b))) if a and b else 0.
def eq(a,b): return -1. if not a or not b else float(a==b)


def structured(a,b):
    n1,n2=a.name,b.name
    v=[float(bool(n1) and n1==n2),float(bool(n1) and bool(n2) and (n1 in n2 or n2 in n1)),SequenceMatcher(None,n1,n2).ratio(),jaccard(a.tokens,b.tokens),contain(a.tokens,b.tokens),min(len(n1),len(n2))/max(1,max(len(n1),len(n2))),min(abs(len(n1)-len(n2))/100,5)]
    for x,y in [(a.brand,b.brand),(a.article,b.article),(a.model,b.model),(a.size,b.size),(a.color,b.color),(a.material,b.material),(a.gender,b.gender)]: v += [eq(x,y),float(bool(x) and bool(y))]
    v += [jaccard(a.quantity,b.quantity),float(bool(a.quantity) and bool(b.quantity) and not a.quantity&b.quantity),float(bool(a.quantity) and bool(b.quantity)),SequenceMatcher(None,a.article,b.article).ratio() if a.article and b.article else 0.,SequenceMatcher(None,a.model,b.model).ratio() if a.model and b.model else 0.]
    return np.asarray(v,np.float32)


# Exact BGE text
BGE_KEYS=["бренд","артикул","партномер","oem","код","модель","размер","цвет","объем","обьем","вес","тип","материал","количество"]
CYR2LAT=str.maketrans("аеорсухАЕОРСУХКМТВНЗЅІі","aeopcyxAEOPCYXKMTBH3SIi"); LAT2CYR=str.maketrans("aeopcyxAEOPCYX","аеорсухАЕОРСУХ")
BUNIT_RE=re.compile(r"(\d+(?:[.,]\d+)?)\s*(мл|ml|л|l|мг|mg|г|g|гр|кг|kg|мм|mm|см|cm|м|мб|mb|гб|gb|тб|tb|вт|w|квт|kw|мач|mah)\b",re.I)
BMUL={"мл":("ml",1),"ml":("ml",1),"л":("ml",1000),"l":("ml",1000),"мг":("g",.001),"mg":("g",.001),"г":("g",1),"g":("g",1),"гр":("g",1),"кг":("g",1000),"kg":("g",1000),"мм":("mm",1),"mm":("mm",1),"см":("mm",10),"cm":("mm",10),"м":("mm",1000),"мб":("gb",.001),"mb":("gb",.001),"гб":("gb",1),"gb":("gb",1),"тб":("gb",1024),"tb":("gb",1024),"вт":("w",1),"w":("w",1),"квт":("w",1000),"kw":("w",1000),"мач":("mah",1),"mah":("mah",1)}
QRES=[re.compile(r"(\d+)\s*шт"),re.compile(r"набор\w*\s+из\s+(\d+)"),re.compile(r"(\d+)\s*(?:набор|упаков|комплект)\w*\s+по\s+(\d+)"),re.compile(r"[xх*](\d+)\b")]


def bge_text(name,attributes):
    parts=[str(name) if name is not None else ""]
    try: d=json.loads(attributes) if isinstance(attributes,str) else {}
    except Exception: d={}
    if isinstance(d,dict) and d:
        low={str(k).lower():str(v) for k,v in d.items() if v}; picked=[]; used=set()
        for wanted in BGE_KEYS:
            for k,v in low.items():
                if wanted in k and k not in used: picked.append(f"{k}:{v}"); used.add(k)
        parts.append(" ; ".join(picked+[f"{k}:{v}" for k,v in low.items() if k not in used])[:520])
    base=" | ".join(parts).replace("ё","е").replace("Ё","Е"); fixed=[]
    for token in base.split():
        c=sum("\u0400"<=x<="\u04ff" for x in token); l=sum(x.isascii() and x.isalpha() for x in token)
        fixed.append(token.translate(CYR2LAT if l>=c else LAT2CYR) if c and l else token)
    base=re.sub(r"[×хХ](?=\d)","x"," ".join(fixed)); extras=[]; units=set()
    for m in BUNIT_RE.finditer(base):
        u,k=BMUL[m.group(2).lower()]; units.add(f"{float(m.group(1).replace(',','.'))*k:g}{u}")
    if units: extras.append("ед: "+" ".join(sorted(units)[:12]))
    q=None; m=QRES[2].search(base.lower())
    if m: q=int(m.group(1))*int(m.group(2))
    else:
        for r in (QRES[0],QRES[1],QRES[3]):
            m=r.search(base.lower())
            if m: q=int(m.group(1)); break
    if q and 1<q<=1000: extras.append(f"кол-во: {q}")
    return (base+(" | "+" | ".join(extras) if extras else ""))[:2000]


human=pl.read_parquet(MATCHES,columns=["id1","id2","target"])
untouched=component_mask(human["id1"].to_numpy(),human["id2"].to_numpy(),.20,42)
val=human.filter(pl.Series(untouched)); assert len(val)==72_948
id1,id2=val["id1"].to_numpy(),val["id2"].to_numpy(); need=set(id1)|set(id2)


def load_items():
    pf=pq.ParquetFile(ITEMS); value_set=pa.array(list(need),type=pf.schema_arrow.field("id").type); ei={}; bt={}; started=time.time()
    for batch in pf.iter_batches(columns=["id","name","attributes","category"],batch_size=300_000,use_threads=True):
        ids=batch.column(batch.schema.get_field_index("id")); selected=batch.filter(pc.is_in(ids,value_set=value_set))
        for i,n,a,c in selected.to_pandas().itertuples(index=False,name=None):
            if i in need: ei[i]=Item(n,a,c); bt[i]=bge_text(n,a)
        if len(ei)==len(need): break
    assert len(ei)==len(need),(len(ei),len(need)); print(f"items={len(ei):,}, {time.time()-started:.1f}s")
    return ei,bt


items,btexts=load_items(); categories=np.asarray([items[x].category for x in id1]); y=val["target"].to_numpy().astype(np.int8)


class HybridStageB(nn.Module):
    def __init__(self,base,ncat,nstruct):
        super().__init__(); self.base=base; self.encoder=base.roberta if hasattr(base,"roberta") else base.xlm_roberta; h=int(base.config.hidden_size)
        self.category_embedding=nn.Embedding(ncat,24); self.category_residual=nn.Sequential(nn.Linear(h+24,128),nn.GELU(),nn.Dropout(.1),nn.Linear(128,1))
        self.structured_residual=nn.Sequential(nn.LayerNorm(nstruct),nn.Linear(nstruct,64),nn.GELU(),nn.Dropout(.1),nn.Linear(64,1))
        self.auxiliary_head=nn.Sequential(nn.Linear(h,128),nn.GELU(),nn.Dropout(.1),nn.Linear(128,6))
    def forward(self,input_ids,attention_mask=None,token_type_ids=None,category_ids=None,structured=None,**kw):
        z={"input_ids":input_ids,"attention_mask":attention_mask,"return_dict":True}
        if token_type_ids is not None:z["token_type_ids"]=token_type_ids
        seq=self.encoder(**z).last_hidden_state; pooled=seq[:,0]
        return self.base.classifier(seq).view(-1)+self.category_residual(torch.cat([pooled,self.category_embedding(category_ids)],-1)).view(-1)+self.structured_residual(structured.float()).view(-1),self.auxiliary_head(pooled)


@torch.inference_mode()
def predict_e5():
    cache=OUT/"e5_v53_manual_val.npy"
    if cache.exists(): return np.load(cache)
    head=torch.load(E5_DIR/"hybrid_heads.pt",map_location="cpu",weights_only=False); cats=[str(x) for x in head["categories"]]; cmap={c:i for i,c in enumerate(cats)}; maxlen=int(head["max_length"])
    assert head.get("pair_mode")=="plain_v2",head.get("pair_mode")
    unknown=sorted(set(categories)-set(cmap)); assert not unknown,f"Unknown categories: {unknown}"
    base=AutoModelForSequenceClassification.from_pretrained(E5_DIR,local_files_only=True,torch_dtype=torch.float32)
    core=HybridStageB(base,len(cats),int(head["n_structured_features"])); incompatible=core.load_state_dict(head["state_dict"],strict=False)
    illegal=[x for x in incompatible.missing_keys if not x.startswith(("base.","encoder."))]
    assert not incompatible.unexpected_keys and not illegal,(incompatible.unexpected_keys,illegal)
    core.to(DEV).eval(); model=nn.DataParallel(core,device_ids=[0,1])
    try: tok=AutoTokenizer.from_pretrained(E5_DIR,local_files_only=True,use_fast=True)
    except (KeyError,ValueError,OSError): tok=PreTrainedTokenizerFast(tokenizer_file=str(E5_DIR/"tokenizer.json"),bos_token="<s>",cls_token="<s>",eos_token="</s>",sep_token="</s>",unk_token="<unk>",pad_token="<pad>",mask_token="<mask>",model_max_length=512)
    order=np.argsort([len(items[a].text)+len(items[b].text) for a,b in zip(id1,id2)],kind="stable"); sorted_pred=np.empty(len(val),np.float32); bs=24; started=time.time()
    for start in range(0,len(order),bs):
        pos=order[start:start+bs]; ab=[(items[id1[i]].text,items[id2[i]].text) for i in pos]; pairs=ab+[(b,a) for a,b in ab]
        enc=tok([a for a,b in pairs],[b for a,b in pairs],padding=True,truncation=True,max_length=maxlen,return_tensors="pt"); enc={k:v.to(DEV,non_blocking=True) for k,v in enc.items()}
        st=np.stack([structured(items[id1[i]],items[id2[i]]) for i in pos]); st=np.concatenate([st,st]); ci=np.asarray([cmap[categories[i]] for i in pos]*2,np.int64)
        with torch.autocast("cuda",dtype=torch.float16): logits,_=model(**enc,structured=torch.from_numpy(st).to(DEV),category_ids=torch.from_numpy(ci).to(DEV))
        n=len(pos); prob=torch.sigmoid(logits.float()); sorted_pred[start:start+n]=((prob[:n]+prob[n:])/2).cpu().numpy()
        if start//bs%500==0: print(f"E5 {start+n:,}/{len(val):,} {(start+n)/(time.time()-started):.0f}/s")
    pred=np.empty(len(val),np.float32); pred[order]=sorted_pred; np.save(cache,pred)
    del model,core,base,tok; gc.collect(); torch.cuda.empty_cache(); return pred


@torch.inference_mode()
def predict_bge():
    cache=OUT/"bge_manual_val.npy"
    if cache.exists(): return np.load(cache)
    tok=AutoTokenizer.from_pretrained(BGE_DIR,local_files_only=True); core=AutoModelForSequenceClassification.from_pretrained(BGE_DIR,local_files_only=True,torch_dtype=torch.float16).to(DEV).eval(); model=nn.DataParallel(core,device_ids=[0,1])
    order=np.argsort([len(btexts[a])+len(btexts[b]) for a,b in zip(id1,id2)],kind="stable"); pred=np.empty(len(val),np.float32); bs=512; started=time.time()
    for start in range(0,len(order),bs):
        pos=order[start:start+bs]; a=[btexts[id1[i]] for i in pos]; b=[btexts[id2[i]] for i in pos]; total=0
        for x,z in ((a,b),(b,a)):
            enc=tok(x,z,padding=True,truncation=True,max_length=320,pad_to_multiple_of=8,return_tensors="pt"); enc={k:v.to(DEV,non_blocking=True) for k,v in enc.items()}
            with torch.autocast("cuda",dtype=torch.float16): total+=torch.sigmoid(model(**enc).logits.squeeze(-1).float()).cpu().numpy()
        pred[pos]=total/2
        if start//bs%40==0: print(f"BGE {start+len(pos):,}/{len(val):,} {(start+len(pos))/(time.time()-started):.0f}/s")
    np.save(cache,pred); del model,core,tok; gc.collect(); torch.cuda.empty_cache(); return pred


e5,bge=predict_e5(),predict_bge(); print("scores ready",len(e5),len(bge))


def rank(x): return pd.Series(x).rank(method="average",pct=True).to_numpy()
def selective(weights):
    out=np.empty(len(e5)); scalar=np.isscalar(weights)
    for c in np.unique(categories):
        ix=np.flatnonzero(categories==c); n=len(ix); out[ix]=rank(e5[ix]); k=max(1,int(np.ceil(n*TOP_FRAC))); chosen=ix[np.argsort(e5[ix],kind="stable")[-k:]]; w=float(weights if scalar else weights.get(str(c),0))
        mixed=(1-w)*rank(e5[chosen])+w*rank(bge[chosen]); out[chosen]=(n-k)/n+rank(mixed)*k/n
    return out
def ap(p,m): return average_precision_score(y[m],p[m]) if len(np.unique(y[m]))>1 else np.nan
def macro(p,m=None):
    if m is None:m=np.ones(len(y),bool)
    return float(np.nanmean([ap(p,m&(categories==c)) for c in np.unique(categories[m])]))


tune=component_mask(id1,id2,.50,2026); evaluation=~tune; ti=np.flatnonzero(tune); inner=component_mask(id1[tune],id2[tune],.50,2027); fit=np.zeros(len(y),bool); check=np.zeros(len(y),bool); fit[ti[~inner]]=1; check[ti[inner]]=1
preds={float(w):selective(float(w)) for w in WEIGHT_GRID}; e5_fit,e5_check=macro(e5,fit),macro(e5,check)
grows=[]
for w,p in preds.items(): grows.append({"weight":w,"fit":macro(p,fit),"check":macro(p,check),"tune":macro(p,tune),"eval":macro(p,evaluation),"full":macro(p)})
gdf=pd.DataFrame(grows); gdf["mean_gain"]=(gdf.fit-e5_fit+gdf.check-e5_check)/2; stable=gdf[(gdf.fit>=e5_fit-.0002)&(gdf.check>=e5_check-.0002)]; best=stable.sort_values("mean_gain",ascending=False).iloc[0]; global_w=float(best.weight) if best.mean_gain>=.0003 else 0.; global_pred=preds[global_w]


weights={}; rows=[]
for c in np.unique(categories):
    cm=categories==c; mf,mc,me=cm&fit,cm&check,cm&evaluation; bf,bc=ap(global_pred,mf),ap(global_pred,mc); candidates=[]
    enough=mf.sum()>=100 and mc.sum()>=100 and y[mf].sum()>=20 and y[mc].sum()>=20 and len(np.unique(y[mf]))>1 and len(np.unique(y[mc]))>1
    if enough:
        for w,p in preds.items():
            fg,cg=ap(p,mf)-bf,ap(p,mc)-bc; candidates.append((w,fg,cg,(fg+cg)/2))
        stable=[r for r in candidates if r[1]>=-.0002 and r[2]>=-.0002 and r[3]>=.0005]; chosen=max(stable,key=lambda r:r[3]) if stable else (global_w,0,0,0)
    else: chosen=(global_w,0,0,0)
    weights[str(c)]=float(chosen[0]); cp=preds[float(chosen[0])]
    rows.append({"category":str(c),"pairs":int(cm.sum()),"positives":int(y[cm].sum()),"weight":float(chosen[0]),"fit_gain":chosen[1],"check_gain":chosen[2],"eval_gain":ap(cp,me)-ap(global_pred,me)})


cat_pred=selective(weights); metrics={"e5":{"tune":macro(e5,tune),"eval":macro(e5,evaluation),"full":macro(e5)},"global":{"weight":global_w,"tune":macro(global_pred,tune),"eval":macro(global_pred,evaluation),"full":macro(global_pred)},"category":{"tune":macro(cat_pred,tune),"eval":macro(cat_pred,evaluation),"full":macro(cat_pred)}}
recommended=metrics["category"]["eval"]>metrics["global"]["eval"]+.0003 and metrics["category"]["full"]>metrics["global"]["full"]
payload={"model":"E5 V5.3 + BGE-M3","top_fraction":TOP_FRAC,"default_bge_weight":global_w,"category_weights":weights,"metrics":metrics,"recommended":recommended}
cat_df=pd.DataFrame(rows).sort_values("eval_gain",ascending=False); cat_df.to_csv(OUT/"category_results.csv",index=False); gdf.to_csv(OUT/"global_grid.csv",index=False); (OUT/"category_weights.json").write_text(json.dumps(payload,ensure_ascii=False,indent=2))
archive=shutil.make_archive("/kaggle/working/v53_category_weight_results","zip",OUT)
print("\nCATEGORY WEIGHTS:\n",cat_df[["category","pairs","positives","weight","fit_gain","check_gain","eval_gain"]].to_string(index=False))
print("\nMETRICS:\n",json.dumps(metrics,ensure_ascii=False,indent=2)); print("\nVERDICT:","CATEGORY WEIGHTS" if recommended else "GLOBAL WEIGHT",global_w); print("saved:",archive)


GPU: ['Tesla T4', 'Tesla T4']
E5 V5.3: /kaggle/input/models/danilzhukovv/ecup-product-matching-e5-v5-3/pytorch/default/1 
BGE: /kaggle/input/models/danilzhukovv/ecup-product-matching-bge/pytorch/default/1


`torch_dtype` is deprecated! Use `dtype` instead!


items=141,906, 79.6s


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

E5 24/72,948 14/s
E5 12,024/72,948 140/s
E5 24,024/72,948 131/s
E5 36,024/72,948 123/s
E5 48,024/72,948 117/s
E5 60,024/72,948 113/s
E5 72,024/72,948 110/s


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

BGE 512/72,948 229/s
BGE 20,992/72,948 115/s
BGE 41,472/72,948 87/s
BGE 61,952/72,948 78/s
scores ready 72948 72948

CATEGORY WEIGHTS:
                category  pairs  positives  weight  fit_gain  check_gain  eval_gain
      Ювелирные изделия   3721        468    0.35  0.000312    0.012566   0.012756
Галантерея и аксессуары   3597        629    0.50  0.004411    0.016825   0.006134
    Товары для животных   3604       1303    0.50  0.003893    0.000497   0.002910
        Бытовая техника   3653       1325    0.50 -0.000197    0.003034   0.001094
       Продукты питания   3665        973    0.05  0.002479    0.000070   0.000252
          Бытовая химия   3426       1613    0.00  0.000000    0.000000   0.000000
             Автотовары   3779        654    0.00  0.000000    0.000000   0.000000
                 Аптека   3630        272    0.00  0.000000    0.000000   0.000000
                 Мебель   3665        584    0.00  0.000000    0.000000   0.000000
         Детские товары   3478    

In [3]:
import contextlib, gc, json, os, re, shutil, time, warnings
from difflib import SequenceMatcher
from pathlib import Path

os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
os.environ["TOKENIZERS_PARALLELISM"]="false"; warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import polars as pl
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
import torch
import torch.nn as nn
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import average_precision_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer, PreTrainedTokenizerFast

START=time.time(); ROOT=Path("/kaggle/input")
BASE=Path("/kaggle/input/datasets/mihailivanovvvv/hakaton-ozon-math-items")
def locate_file(expected,name,prefer=""):
    if expected.exists():return expected
    hits=list(ROOT.rglob(name))
    if prefer:hits.sort(key=lambda p:(prefer.lower() not in str(p).lower(),len(str(p))))
    else:hits.sort(key=lambda p:len(str(p)))
    if not hits:raise FileNotFoundError(f"Не найден {name} в /kaggle/input. Подключи нужный Dataset.")
    return hits[0]
HUMAN=locate_file(BASE/"matches.parquet","matches.parquet","hakaton-ozon-math-items")
ITEMS_H=locate_file(BASE/"items_human.parquet","items_human.parquet","hakaton-ozon-math-items")
LLM_FULL=locate_file(BASE/"matches_llm.parquet","matches_llm.parquet","hakaton-ozon-math-items")
ITEMS=locate_file(BASE/"items.parquet","items.parquet","hakaton-ozon-math-items")
POLYGON=locate_file(Path("/kaggle/input/datasets/w4stdd/2m-parquet/llm_sample_2m.parquet"),"llm_sample_2m.parquet","2m-parquet")
WORK=Path("/kaggle/working/catboost_residual_bank"); WORK.mkdir(exist_ok=True)
CACHE=WORK/"cache"; CACHE.mkdir(exist_ok=True); MODELS=WORK/"models"; MODELS.mkdir(exist_ok=True)
EXPORT=WORK/"export"; EXPORT.mkdir(exist_ok=True)
DEV=torch.device("cuda:0"); GPUS=list(range(torch.cuda.device_count())); DEVICES="0:1" if len(GPUS)>1 else "0"
assert len(GPUS)==2,f"Нужно 2 GPU, сейчас видно: {len(GPUS)}"
print("INPUTS:")
for label,path in {"human":HUMAN,"items_human":ITEMS_H,"llm_full":LLM_FULL,"items_full":ITEMS,"polygon_2m":POLYGON}.items():print(f"  {label}: {path}")

E5_FIXED=Path("/kaggle/input/models/danilzhukovv/ecup-product-matching-e5/pytorch/default/1")
BGE_FIXED=Path("/kaggle/input/models/danilzhukovv/ecup-product-matching-bge/pytorch/default/1")
def find_e5():
    if (E5_FIXED/"hybrid_heads.pt").exists(): return E5_FIXED
    hits=[p.parent for p in ROOT.rglob("hybrid_heads.pt") if "v5-3" not in str(p).lower() and "v5_3" not in str(p).lower()]
    if not hits: raise FileNotFoundError("Не найдена E5 V5.2 с hybrid_heads.pt")
    return sorted(hits,key=lambda p:len(str(p)))[0]
def find_bge():
    if (BGE_FIXED/"model.safetensors").exists(): return BGE_FIXED
    for p in ROOT.rglob("config.json"):
        try:
            c=json.loads(p.read_text())
            if c.get("hidden_size")==1024 and c.get("num_hidden_layers")==24 and (p.parent/"model.safetensors").exists(): return p.parent
        except: pass
    raise FileNotFoundError("Не найдена BGE-M3")
E5_DIR,BGE_DIR=find_e5(),find_bge()
print("GPU:",[torch.cuda.get_device_name(i) for i in GPUS]); print("E5 V5.2:",E5_DIR,"\nBGE:",BGE_DIR)

# ---------- component-safe datasets ----------
def components(a,b):
    parent={}
    def find(x):
        parent.setdefault(x,x)
        while parent[x]!=x: parent[x]=parent[parent[x]]; x=parent[x]
        return x
    for x,y in zip(a,b):
        x,y=find(x),find(y)
        if x!=y: parent[y]=x
    return np.fromiter((find(x) for x in a),np.int64,len(a))
def group_mask(comp,frac,seed):
    groups=np.unique(comp); rng=np.random.RandomState(seed); chosen=set(groups[rng.rand(len(groups))<frac])
    return np.fromiter((x in chosen for x in comp),bool,len(comp))

HP=CACHE/"human_untouched.parquet"; LP=CACHE/"llm_holdout.parquet"; TP=CACHE/"llm_train_2m.parquet"
if not HP.exists():
    h=pl.read_parquet(HUMAN,columns=["id1","id2","target"]).to_pandas(); comp=components(h.id1.to_numpy(),h.id2.to_numpy()); m=group_mask(comp,.20,42)
    h=h.loc[m].copy(); h["component"]=comp[m]; h.to_parquet(HP,index=False); print("human untouched:",len(h))
if not LP.exists():
    t=time.time(); llm=pl.read_parquet(LLM_FULL,columns=["id1","id2","target"]).to_pandas(); comp=components(llm.id1.to_numpy(),llm.id2.to_numpy()); m=group_mask(comp,.03,13)
    llm=llm.loc[m].copy(); val_comp=comp[m]; clean=((llm.target<=.2)|(llm.target>=.8)).to_numpy(); llm=llm.loc[clean].copy(); llm["target_soft"]=llm.target; llm["target"]=(llm.target>=.5).astype(np.int8); llm["component"]=val_comp[clean]
    llm.to_parquet(LP,index=False); print(f"exact LLM holdout: {len(llm):,}, {(time.time()-t)/60:.1f} min")
if not TP.exists():
    hold=pl.read_parquet(LP,columns=["id1","id2"]).with_columns(pl.min_horizontal("id1","id2").alias("_a"),pl.max_horizontal("id1","id2").alias("_b")).select("_a","_b").unique()
    train=(pl.read_parquet(POLYGON,columns=["id1","id2","target"])
           .with_columns(pl.min_horizontal("id1","id2").alias("_a"),pl.max_horizontal("id1","id2").alias("_b"))
           .join(hold,on=["_a","_b"],how="anti").drop("_a","_b"))
    if len(train)>2_000_000: train=train.sample(n=2_000_000,seed=314159,shuffle=True)
    train.write_parquet(TP,compression="zstd"); print("LLM soft train:",len(train))
h=pd.read_parquet(HP); lh=pd.read_parquet(LP); lt=pd.read_parquet(TP)
assert len(h)==72_948 and len(lh)==191_555

# ---------- exact V5 structured features ----------
SPACE_RE=re.compile(r"\s+"); MULTIPLY_RE=re.compile(r"[×хХ*]"); TOKEN_RE=re.compile(r"[0-9a-zа-я]+",re.I)
SIZE_RE=re.compile(r"(?:размер|р-р|size)\s*[:=\-]?\s*([0-9]{2,3}(?:[./-][0-9]{1,3})?|xxxs|xxs|xs|s|m|l|xl|xxl|xxxl)",re.I)
UNIT_RE=re.compile(r"(\d+(?:[.,]\d+)?)\s*(мл|ml|л|l|мг|mg|г|gr|g|кг|kg|мм|mm|см|cm|м|m|вт|w|квт|kw|шт|pcs|pack)\b",re.I)
PACK_RE=re.compile(r"(\d+)\s*[xх×*]\s*(\d+(?:[.,]\d+)?)\s*(мл|ml|л|l|мг|mg|г|gr|g|кг|kg|шт|pcs)",re.I)
KEY_ORDER=["бренд","brand","производитель","артикул","партномер","part number","partnumber","oem","код","sku","модель","model","размер","size","рост","обхват","пол","gender","цвет","color","материал","material","объем","обьем","volume","вес","weight","количество","комплектация","упаков","тип","type"]
ALIASES={"brand":["бренд","brand","производитель"],"article":["артикул","sku","oem","партномер","part number","partnumber","код производителя"],"model":["модель","model"],"size":["размер производителя","размер обуви","размер одежды","размер","size","рост"],"color":["основной цвет","цвет","color","расцветка"],"material":["материал верха","материал","material"],"gender":["пол","gender"]}
HOMO=str.maketrans({"а":"a","в":"b","с":"c","е":"e","н":"h","к":"k","м":"m","о":"o","р":"p","т":"t","х":"x","у":"y"})
COLORS={x.replace("ё","е"):y for x,y in {"черный":"черный","чёрный":"черный","белый":"белый","серый":"серый","серебристый":"серебристый","красный":"красный","бордовый":"бордовый","синий":"синий","голубой":"голубой","зеленый":"зеленый","зелёный":"зеленый","желтый":"желтый","жёлтый":"желтый","оранжевый":"оранжевый","розовый":"розовый","фиолетовый":"фиолетовый","бежевый":"бежевый","коричневый":"коричневый","золотой":"золотой","золотистый":"золотой"}.items()}
def norm(x): return "" if x is None else SPACE_RE.sub(" ",MULTIPLY_RE.sub("x",str(x).lower().replace("ё","е").replace(",","."))).strip()
def attrs_dict(x):
    if isinstance(x,dict): d=x
    elif isinstance(x,str):
        try:d=json.loads(x)
        except:d={}
    else:d={}
    return {norm(k):norm(v) for k,v in d.items() if norm(k) and norm(v)} if isinstance(d,dict) else {}
def pick(d,a):
    for alias in a:
        for k,v in d.items():
            if alias in k:return v
    return ""
def compact(x,limit=96):return SPACE_RE.sub(" ",re.sub(r"[^0-9a-zа-я.+/_\- ]+"," ",norm(x))).strip()[:limit]
def code_norm(x):return re.sub(r"[^0-9a-z]+","",compact(x).translate(HOMO))
def size_norm(x):return re.sub(r"\s*/\s*","/",re.sub(r"\s*-\s*","-",compact(x,64).replace("–","-").replace("—","-")))
def color_norm(x):
    text=compact(x,64); found=[v for k,v in COLORS.items() if k in text]; return "/".join(sorted(set(found))) if found else text
def fallback_color(x):
    text=norm(x); return "/".join(sorted(set(v for k,v in COLORS.items() if k in text)))
def quantity_tokens(text):
    def cv(v,u):
        u=u.lower()
        if u in {"мл","ml"}:return "volume_ml",v
        if u in {"л","l"}:return "volume_ml",v*1000
        if u in {"мг","mg"}:return "mass_g",v/1000
        if u in {"г","gr","g"}:return "mass_g",v
        if u in {"кг","kg"}:return "mass_g",v*1000
        if u in {"мм","mm"}:return "length_mm",v
        if u in {"см","cm"}:return "length_mm",v*10
        if u in {"м","m"}:return "length_mm",v*1000
        if u in {"вт","w"}:return "power_w",v
        if u in {"квт","kw"}:return "power_w",v*1000
        return "count",v
    text=norm(text); out=set()
    for c,a,u in PACK_RE.findall(text): kind,v=cv(float(a.replace(",",".")),u); out|={f"{kind}:{round(float(c)*v,4)}",f"pack_count:{round(float(c),4)}"}
    for a,u in UNIT_RE.findall(text): kind,v=cv(float(a.replace(",",".")),u); out.add(f"{kind}:{round(v,4)}")
    return frozenset(out)
class Item:
    __slots__=("name","tokens","brand","article","model","size","color","material","gender","quantity","category")
    def __init__(self,name,attributes,category):
        d=attrs_dict(attributes); self.name=norm(name); self.tokens=frozenset(x for x in TOKEN_RE.findall(self.name) if len(x)>=2); self.brand=compact(pick(d,ALIASES["brand"])); self.article=code_norm(pick(d,ALIASES["article"])); self.model=code_norm(pick(d,ALIASES["model"])); m=SIZE_RE.search(self.name); self.size=size_norm(pick(d,ALIASES["size"])) or (size_norm(m.group(1)) if m else ""); self.color=color_norm(pick(d,ALIASES["color"])) or fallback_color(self.name); self.material=compact(pick(d,ALIASES["material"])); self.gender=compact(pick(d,ALIASES["gender"])); self.category=str(category); self.quantity=quantity_tokens(self.name+" "+" ".join(d.values())[:1800])
def jaccard(a,b):return len(a&b)/len(a|b) if a|b else 0.
def contain(a,b):return len(a&b)/max(1,min(len(a),len(b))) if a and b else 0.
def eq(a,b):return -1. if not a or not b else float(a==b)
def structured(a,b):
    n1,n2=a.name,b.name; v=[float(bool(n1) and n1==n2),float(bool(n1) and bool(n2) and (n1 in n2 or n2 in n1)),SequenceMatcher(None,n1,n2).ratio(),jaccard(a.tokens,b.tokens),contain(a.tokens,b.tokens),min(len(n1),len(n2))/max(1,max(len(n1),len(n2))),min(abs(len(n1)-len(n2))/100,5)]
    for x,y in [(a.brand,b.brand),(a.article,b.article),(a.model,b.model),(a.size,b.size),(a.color,b.color),(a.material,b.material),(a.gender,b.gender)]:v += [eq(x,y),float(bool(x) and bool(y))]
    v += [jaccard(a.quantity,b.quantity),float(bool(a.quantity) and bool(b.quantity) and not a.quantity&b.quantity),float(bool(a.quantity) and bool(b.quantity)),SequenceMatcher(None,a.article,b.article).ratio() if a.article and b.article else 0.,SequenceMatcher(None,a.model,b.model).ratio() if a.model and b.model else 0.]
    return np.asarray(v,np.float32)
MANUAL=[f"structured_{i:02d}" for i in range(26)]

def e5_text(name,attributes,category):
    d=attrs_dict(attributes); used=set(); priority=[]
    for wanted in KEY_ORDER:
        for k,v in d.items():
            if k not in used and wanted in k:priority.append(f"{k}: {v}"); used.add(k)
    attrs=" ; ".join(priority+[f"{k}: {v}" for k,v in d.items() if k not in used])[:620]
    return f"категория: {norm(category)} | название: {norm(name)} | атрибуты: {attrs}"
BGE_KEYS=["бренд","артикул","партномер","oem","код","модель","размер","цвет","объем","обьем","вес","тип","материал","количество"]
CYR2LAT=str.maketrans("аеорсухАЕОРСУХКМТВНЗЅІі","aeopcyxAEOPCYXKMTBH3SIi"); LAT2CYR=str.maketrans("aeopcyxAEOPCYX","аеорсухАЕОРСУХ")
BUNIT_RE=re.compile(r"(\d+(?:[.,]\d+)?)\s*(мл|ml|л|l|мг|mg|г|g|гр|кг|kg|мм|mm|см|cm|м|мб|mb|гб|gb|тб|tb|вт|w|квт|kw|мач|mah)\b",re.I)
BMUL={"мл":("ml",1),"ml":("ml",1),"л":("ml",1000),"l":("ml",1000),"мг":("g",.001),"mg":("g",.001),"г":("g",1),"g":("g",1),"гр":("g",1),"кг":("g",1000),"kg":("g",1000),"мм":("mm",1),"mm":("mm",1),"см":("mm",10),"cm":("mm",10),"м":("mm",1000),"мб":("gb",.001),"mb":("gb",.001),"гб":("gb",1),"gb":("gb",1),"тб":("gb",1024),"tb":("gb",1024),"вт":("w",1),"w":("w",1),"квт":("w",1000),"kw":("w",1000),"мач":("mah",1),"mah":("mah",1)}
QRES=[re.compile(r"(\d+)\s*шт"),re.compile(r"набор\w*\s+из\s+(\d+)"),re.compile(r"(\d+)\s*(?:набор|упаков|комплект)\w*\s+по\s+(\d+)"),re.compile(r"[xх*](\d+)\b")]
def bge_text(name,attributes):
    parts=[str(name) if name is not None else ""]
    try:d=json.loads(attributes) if isinstance(attributes,str) else {}
    except:d={}
    if isinstance(d,dict) and d:
        low={str(k).lower():str(v) for k,v in d.items() if v}; picked=[]; used=set()
        for wanted in BGE_KEYS:
            for k,v in low.items():
                if wanted in k and k not in used:picked.append(f"{k}:{v}"); used.add(k)
        parts.append(" ; ".join(picked+[f"{k}:{v}" for k,v in low.items() if k not in used])[:520])
    base=" | ".join(parts).replace("ё","е").replace("Ё","Е"); fixed=[]
    for token in base.split():
        c=sum("\u0400"<=x<="\u04ff" for x in token); l=sum(x.isascii() and x.isalpha() for x in token); fixed.append(token.translate(CYR2LAT if l>=c else LAT2CYR) if c and l else token)
    base=re.sub(r"[×хХ](?=\d)","x"," ".join(fixed)); extras=[]; units=set()
    for m in BUNIT_RE.finditer(base):u,k=BMUL[m.group(2).lower()];units.add(f"{float(m.group(1).replace(',','.'))*k:g}{u}")
    if units:extras.append("ед: "+" ".join(sorted(units)[:12]))
    q=None;m=QRES[2].search(base.lower())
    if m:q=int(m.group(1))*int(m.group(2))
    else:
        for r in (QRES[0],QRES[1],QRES[3]):
            m=r.search(base.lower())
            if m:q=int(m.group(1));break
    if q and 1<q<=1000:extras.append(f"кол-во: {q}")
    return (base+(" | "+" | ".join(extras) if extras else ""))[:2000]

# ---------- profiles + feature cache ----------
HF=CACHE/"human_features.npy"; LF=CACHE/"llm_holdout_features.npy"; TF=CACHE/"llm_train_features.npy"
HC=CACHE/"human_categories.npy"; LC=CACHE/"llm_holdout_categories.npy"; TC=CACHE/"llm_train_categories.npy"; TEXTS=CACHE/"score_texts.parquet"
if not all(p.exists() for p in [HF,LF,TF,HC,LC,TC,TEXTS]):
    need=set(h.id1)|set(h.id2)|set(lh.id1)|set(lh.id2)|set(lt.id1)|set(lt.id2); score_need=set(h.id1)|set(h.id2)|set(lh.id1)|set(lh.id2)
    profiles={}; text_rows=[]; remaining=set(need); t=time.time()
    for item_path in [ITEMS,ITEMS_H]:
        if not remaining:break
        pf=pq.ParquetFile(item_path); values=pa.array(list(remaining),type=pf.schema_arrow.field("id").type)
        for batch in pf.iter_batches(columns=["id","name","attributes","category"],batch_size=300_000,use_threads=True):
            ids=batch.column(batch.schema.get_field_index("id")); selected=batch.filter(pc.is_in(ids,value_set=values))
            for i,n,a,c in selected.to_pandas().itertuples(index=False,name=None):
                if i in remaining:
                    profiles[i]=Item(n,a,c); remaining.remove(i)
                    if i in score_need:text_rows.append((i,e5_text(n,a,c),bge_text(n,a),str(c)))
        print(f"profiles {len(profiles):,}/{len(need):,}")
    assert not remaining,f"Missing items: {len(remaining)}"
    pd.DataFrame(text_rows,columns=["id","e5_text","bge_text","category"]).to_parquet(TEXTS,index=False)
    def build(df,fpath,cpath,label):
        x=np.lib.format.open_memmap(fpath,mode="w+",dtype=np.float32,shape=(len(df),26)); cats=np.empty(len(df),dtype="<U64"); started=time.time(); a=df.id1.to_numpy(); b=df.id2.to_numpy()
        for s in range(0,len(df),50_000):
            e=min(s+50_000,len(df))
            for j in range(s,e):x[j]=structured(profiles[a[j]],profiles[b[j]]); cats[j]=profiles[a[j]].category
            print(f"{label} {e:,}/{len(df):,} {e/max(time.time()-started,1):.0f}/s")
        x.flush(); np.save(cpath,cats)
    if not HF.exists():build(h,HF,HC,"human features")
    if not LF.exists():build(lh,LF,LC,"LLM holdout features")
    if not TF.exists():build(lt,TF,TC,"LLM train features")
    del profiles,text_rows; gc.collect(); print(f"feature stage {(time.time()-t)/60:.1f} min")
Xh=np.load(HF,mmap_mode="r"); Xl=np.load(LF,mmap_mode="r"); Xt=np.load(TF,mmap_mode="r"); ch=np.load(HC); cl=np.load(LC); ct=np.load(TC)

# ---------- E5 V5.2 + BGE scores on honest holdouts ----------
EH=CACHE/"e5_human.npy"; EL=CACHE/"e5_llm.npy"; BH=CACHE/"bge_human.npy"; BL=CACHE/"bge_llm.npy"
texts_df=pd.read_parquet(TEXTS); e5map=dict(zip(texts_df.id,texts_df.e5_text)); bgemap=dict(zip(texts_df.id,texts_df.bge_text))
pair_all=pd.concat([h[["id1","id2"]],lh[["id1","id2"]]],ignore_index=True); cats_all=np.concatenate([ch,cl]); struct_all=np.concatenate([np.asarray(Xh),np.asarray(Xl)]); split=len(h)
class HybridStageB(nn.Module):
    def __init__(self,base,ncat,nstruct):
        super().__init__(); self.base=base; self.encoder=base.roberta if hasattr(base,"roberta") else base.xlm_roberta; z=int(base.config.hidden_size); self.category_embedding=nn.Embedding(ncat,24); self.category_residual=nn.Sequential(nn.Linear(z+24,128),nn.GELU(),nn.Dropout(.1),nn.Linear(128,1)); self.structured_residual=nn.Sequential(nn.LayerNorm(nstruct),nn.Linear(nstruct,64),nn.GELU(),nn.Dropout(.1),nn.Linear(64,1)); self.auxiliary_head=nn.Sequential(nn.Linear(z,128),nn.GELU(),nn.Dropout(.1),nn.Linear(128,6))
    def forward(self,input_ids,attention_mask=None,token_type_ids=None,category_ids=None,structured=None,**kw):
        z={"input_ids":input_ids,"attention_mask":attention_mask,"return_dict":True}
        if token_type_ids is not None:z["token_type_ids"]=token_type_ids
        seq=self.encoder(**z).last_hidden_state; pooled=seq[:,0]; return self.base.classifier(seq).view(-1)+self.category_residual(torch.cat([pooled,self.category_embedding(category_ids)],-1)).view(-1)+self.structured_residual(structured.float()).view(-1),self.auxiliary_head(pooled)
@torch.inference_mode()
def predict_e5():
    if EH.exists() and EL.exists():return np.load(EH),np.load(EL)
    head=torch.load(E5_DIR/"hybrid_heads.pt",map_location="cpu",weights_only=False); categories=[str(x) for x in head["categories"]]; cmap={c:i for i,c in enumerate(categories)}; assert head.get("pair_mode")=="plain_v2"
    base=AutoModelForSequenceClassification.from_pretrained(E5_DIR,local_files_only=True,dtype=torch.float32); core=HybridStageB(base,len(categories),int(head["n_structured_features"])); inc=core.load_state_dict(head["state_dict"],strict=False); assert not inc.unexpected_keys; core.to(DEV).eval(); model=nn.DataParallel(core,device_ids=GPUS)
    try:tok=AutoTokenizer.from_pretrained(E5_DIR,local_files_only=True,use_fast=True)
    except:tok=PreTrainedTokenizerFast(tokenizer_file=str(E5_DIR/"tokenizer.json"),bos_token="<s>",cls_token="<s>",eos_token="</s>",sep_token="</s>",unk_token="<unk>",pad_token="<pad>",mask_token="<mask>",model_max_length=512)
    a=pair_all.id1.to_numpy(); b=pair_all.id2.to_numpy(); order=np.argsort([len(e5map[x])+len(e5map[y]) for x,y in zip(a,b)],kind="stable"); sorted_pred=np.empty(len(pair_all),np.float32); bs=24; started=time.time()
    for s in range(0,len(order),bs):
        pos=order[s:s+bs]; ab=[(e5map[a[i]],e5map[b[i]]) for i in pos]; pairs=ab+[(y,x) for x,y in ab]; enc=tok([x for x,y in pairs],[y for x,y in pairs],padding=True,truncation=True,max_length=int(head["max_length"]),return_tensors="pt"); enc={k:v.to(DEV,non_blocking=True) for k,v in enc.items()}; st=np.concatenate([struct_all[pos],struct_all[pos]]); ci=np.asarray([cmap[str(cats_all[i])] for i in pos]*2,np.int64)
        with torch.autocast("cuda",dtype=torch.float16):logits,_=model(**enc,structured=torch.from_numpy(st).to(DEV),category_ids=torch.from_numpy(ci).to(DEV))
        n=len(pos); p=torch.sigmoid(logits.float()); sorted_pred[s:s+n]=((p[:n]+p[n:])/2).cpu().numpy()
        if s//bs%1000==0:print(f"E5 {s+n:,}/{len(order):,} {(s+n)/max(time.time()-started,1):.0f}/s")
    pred=np.empty(len(order),np.float32); pred[order]=sorted_pred; np.save(EH,pred[:split]); np.save(EL,pred[split:]); del model,core,base,tok; gc.collect(); torch.cuda.empty_cache(); return pred[:split],pred[split:]
@torch.inference_mode()
def predict_bge():
    if BH.exists() and BL.exists():return np.load(BH),np.load(BL)
    tok=AutoTokenizer.from_pretrained(BGE_DIR,local_files_only=True); core=AutoModelForSequenceClassification.from_pretrained(BGE_DIR,local_files_only=True,dtype=torch.float16).to(DEV).eval(); model=nn.DataParallel(core,device_ids=GPUS); a=pair_all.id1.to_numpy(); b=pair_all.id2.to_numpy(); order=np.argsort([len(bgemap[x])+len(bgemap[y]) for x,y in zip(a,b)],kind="stable"); pred=np.empty(len(pair_all),np.float32); bs=512; started=time.time()
    for s in range(0,len(order),bs):
        pos=order[s:s+bs]; x=[bgemap[a[i]] for i in pos]; y=[bgemap[b[i]] for i in pos]; total=0
        for u,v in ((x,y),(y,x)):
            enc=tok(u,v,padding=True,truncation=True,max_length=320,pad_to_multiple_of=8,return_tensors="pt"); enc={k:v.to(DEV,non_blocking=True) for k,v in enc.items()}
            with torch.autocast("cuda",dtype=torch.float16):total+=torch.sigmoid(model(**enc).logits.squeeze(-1).float()).cpu().numpy()
        pred[pos]=total/2
        if s//bs%40==0:print(f"BGE {s+len(pos):,}/{len(order):,} {(s+len(pos))/max(time.time()-started,1):.0f}/s")
    np.save(BH,pred[:split]); np.save(BL,pred[split:]); del model,core,tok; gc.collect(); torch.cuda.empty_cache(); return pred[:split],pred[split:]
e5h,e5l=predict_e5(); bgeh,bgel=predict_bge(); del pair_all,struct_all,texts_df,e5map,bgemap; gc.collect()

# ---------- metrics/rank helpers ----------
def rank(x):return pd.Series(x).rank(method="average",pct=True).to_numpy()
def cat_rank(x,c):
    out=np.empty(len(x),np.float64)
    for z in np.unique(c):m=c==z; out[m]=rank(np.asarray(x)[m])
    return out
def macro(y,p,c,m=None):
    if m is None:m=np.ones(len(y),bool)
    return float(np.mean([average_precision_score(y[m&(c==z)],p[m&(c==z)]) for z in np.unique(c[m])]))
def selective(e5,bge,c,frac=.10,w=.20):
    out=cat_rank(e5,c)
    for z in np.unique(c):
        ix=np.flatnonzero(c==z); k=max(1,int(np.ceil(len(ix)*frac))); chosen=ix[np.argsort(e5[ix],kind="stable")[-k:]]; mixed=(1-w)*rank(e5[chosen])+w*rank(bge[chosen]); out[chosen]=(len(ix)-k)/len(ix)+rank(mixed)*k/len(ix)
    return out
def blend(base,extra,c,a):return (1-a)*cat_rank(base,c)+a*cat_rank(extra,c)
yh=h.target.to_numpy(np.int8); yl=lh.target.to_numpy(np.int8); baseh=selective(e5h,bgeh,ch); basel=selective(e5l,bgel,cl)
print("\nBASELINES manual full:",{"E5_V5.2":macro(yh,e5h,ch),"BGE":macro(yh,bgeh,ch),"old_selective":macro(yh,baseh,ch)})
print("BASELINES exact LLM:",{"E5_V5.2":macro(yl,e5l,cl),"BGE":macro(yl,bgel,cl),"old_selective":macro(yl,basel,cl)})

# ---------- 8 LLM-soft CatBoost teachers ----------
LLM_CONFIGS=[
    {"name":"llm_d6_s11","depth":6,"lr":.035,"l2":6,"seed":11},{"name":"llm_d7_s22","depth":7,"lr":.03,"l2":8,"seed":22},
    {"name":"llm_d8_s33","depth":8,"lr":.025,"l2":10,"seed":33},{"name":"llm_d9_s44","depth":9,"lr":.022,"l2":12,"seed":44},
    {"name":"llm_d6_s55","depth":6,"lr":.025,"l2":14,"seed":55},{"name":"llm_d7_s66","depth":7,"lr":.02,"l2":18,"seed":66},
    {"name":"llm_d8_s77","depth":8,"lr":.02,"l2":20,"seed":77},{"name":"llm_d10_s88","depth":10,"lr":.015,"l2":24,"seed":88}]
def frame(x,c):
    d=pd.DataFrame(np.asarray(x),columns=MANUAL,copy=False); d.insert(0,"category",np.asarray(c).astype(str)); return d
Ft=frame(Xt,ct); Fh_manual=frame(Xh,ch); Fl_manual=frame(Xl,cl); yt=lt.target.to_numpy(np.float32)
rng=np.random.RandomState(730); perm=rng.permutation(len(yt)); cut=int(len(yt)*.92); tr,va=perm[:cut],perm[cut:]
counts=pd.Series(ct).value_counts(); med=float(counts.median()); cw={k:float(np.clip((med/v)**.5,.65,1.7)) for k,v in counts.items()}; sw=np.asarray([cw[x] for x in ct],np.float32)*(.65+1.35*np.abs(yt-.5)*2)
train_pool=Pool(Ft.iloc[tr],yt[tr],cat_features=["category"],weight=sw[tr]); val_pool=Pool(Ft.iloc[va],yt[va],cat_features=["category"],weight=sw[va]); full_pool=Pool(Ft,yt,cat_features=["category"],weight=sw)
llm_rows=[]
for cfg in LLM_CONFIGS:
    mp=MODELS/f"{cfg['name']}.cbm"; ph=CACHE/f"{cfg['name']}_human.npy"; plp=CACHE/f"{cfg['name']}_llm.npy"; meta=MODELS/f"{cfg['name']}.json"
    if not (mp.exists() and ph.exists() and plp.exists()):
        p=dict(iterations=4000,depth=cfg["depth"],learning_rate=cfg["lr"],l2_leaf_reg=cfg["l2"],random_strength=1,loss_function="CrossEntropy",eval_metric="CrossEntropy",task_type="GPU",devices=DEVICES,border_count=128,random_seed=cfg["seed"],allow_writing_files=False,verbose=250,od_type="Iter",od_wait=180)
        model=CatBoostClassifier(**p); model.fit(train_pool,eval_set=val_pool,use_best_model=True); best=max(200,model.get_best_iteration()+1); p.pop("od_type");p.pop("od_wait");p["iterations"]=best;p["verbose"]=300
        final=CatBoostClassifier(**p); final.fit(full_pool); final.save_model(mp); np.save(ph,final.predict_proba(Fh_manual)[:,1]); np.save(plp,final.predict_proba(Fl_manual)[:,1]); meta.write_text(json.dumps({**cfg,"best_iteration":best},indent=2)); del model,final;gc.collect();torch.cuda.empty_cache()
    hp=np.load(ph); lp=np.load(plp); llm_rows.append({"name":cfg["name"],"manual":macro(yh,hp,ch),"llm":macro(yl,lp,cl)})
llm_table=pd.DataFrame(llm_rows).sort_values("llm",ascending=False); print("\nLLM SOFT BANK:\n",llm_table.to_string(index=False)); top_llm=llm_table.head(3).name.tolist(); print("top LLM teachers:",top_llm)
del train_pool,val_pool,full_pool,Ft,sw;gc.collect();torch.cuda.empty_cache()

# ---------- residual meta features ----------
def logit(x):x=np.clip(x,1e-5,1-1e-5);return np.log(x/(1-x))
def meta_frame(manual,c,e5,bge,base,domain_preds):
    d=pd.DataFrame(np.asarray(manual),columns=MANUAL,copy=False); d.insert(0,"category",np.asarray(c).astype(str)); d["e5"]=e5;d["bge"]=bge;d["e5_logit"]=logit(e5);d["bge_logit"]=logit(bge);d["e5_rank"]=cat_rank(e5,c);d["bge_rank"]=cat_rank(bge,c);d["old_selective"]=base;d["score_diff"]=bge-e5;d["rank_diff"]=d.bge_rank-d.e5_rank;d["score_absdiff"]=np.abs(bge-e5)
    for name,p in domain_preds.items():d[f"teacher_{name}"]=p
    a=np.column_stack(list(domain_preds.values()));d["teacher_mean"]=a.mean(1);d["teacher_std"]=a.std(1);d["teacher_min"]=a.min(1);d["teacher_max"]=a.max(1);return d
hteach={n:np.load(CACHE/f"{n}_human.npy") for n in top_llm}; lteach={n:np.load(CACHE/f"{n}_llm.npy") for n in top_llm}
Mh=meta_frame(Xh,ch,e5h,bgeh,baseh,hteach); Ml=meta_frame(Xl,cl,e5l,bgel,basel,lteach); FEATURES=[x for x in Mh.columns if x!="category"]

META_CONFIGS=[
 {"name":"global_d6","kind":"global","depth":6,"iterations":1400,"lr":.025,"l2":8,"hard":False},
 {"name":"global_d7","kind":"global","depth":7,"iterations":1500,"lr":.022,"l2":10,"hard":False},
 {"name":"global_d8","kind":"global","depth":8,"iterations":1600,"lr":.018,"l2":14,"hard":False},
 {"name":"global_d9","kind":"global","depth":9,"iterations":1400,"lr":.018,"l2":18,"hard":False},
 {"name":"global_hard_d7","kind":"global","depth":7,"iterations":1700,"lr":.02,"l2":12,"hard":True},
 {"name":"global_hard_d8","kind":"global","depth":8,"iterations":1600,"lr":.018,"l2":18,"hard":True},
 {"name":"category_d5","kind":"category","depth":5,"iterations":900,"lr":.025,"l2":12,"hard":False},
 {"name":"category_hard_d6","kind":"category","depth":6,"iterations":1100,"lr":.02,"l2":16,"hard":True}]
counts=pd.Series(ch).value_counts(); med=float(counts.median()); category_weight={k:float(np.clip((med/v)**.5,.7,1.5)) for k,v in counts.items()}
def weights(indices,hard):
    w=np.asarray([category_weight[x] for x in ch[indices]],np.float32)
    if hard:
        yy=yh[indices]; ss=e5h[indices]; w*=1+3*((yy==0)&(ss>.55))+1.5*((yy==1)&(ss<.45))
    return w
def fit_candidate(cfg,train_idx,out_frames,seed,save_dir=None):
    preds=[np.empty(len(x),np.float64) for x in out_frames]
    if cfg["kind"]=="global":
        p=dict(iterations=cfg["iterations"],depth=cfg["depth"],learning_rate=cfg["lr"],l2_leaf_reg=cfg["l2"],random_strength=1,loss_function="Logloss",task_type="GPU",devices=DEVICES,border_count=128,random_seed=seed,allow_writing_files=False,verbose=False)
        model=CatBoostClassifier(**p); model.fit(Pool(Mh.iloc[train_idx],yh[train_idx],cat_features=["category"],weight=weights(train_idx,cfg["hard"])))
        for i,x in enumerate(out_frames):preds[i]=model.predict_proba(x)[:,1]
        if save_dir:model.save_model(Path(save_dir)/"meta_global.cbm")
        del model
    else:
        Path(save_dir).mkdir(exist_ok=True,parents=True) if save_dir else None
        for cat in np.unique(ch):
            tr=train_idx[ch[train_idx]==cat]
            if len(tr)<100 or len(np.unique(yh[tr]))<2:raise RuntimeError(f"Too little category train: {cat}")
            p=dict(iterations=cfg["iterations"],depth=cfg["depth"],learning_rate=cfg["lr"],l2_leaf_reg=cfg["l2"],random_strength=1,loss_function="Logloss",task_type="CPU",thread_count=4,random_seed=seed,allow_writing_files=False,verbose=False)
            model=CatBoostClassifier(**p); model.fit(Mh.iloc[tr][FEATURES],yh[tr],sample_weight=weights(tr,cfg["hard"]));
            for i,x in enumerate(out_frames):
                m=x.category.to_numpy()==cat
                if m.any():preds[i][m]=model.predict_proba(x.loc[m,FEATURES])[:,1]
            if save_dir:model.save_model(Path(save_dir)/(re.sub(r"[^0-9a-zа-я]+","_",str(cat).lower()).strip("_")+".cbm"))
            del model
    gc.collect();torch.cuda.empty_cache();return preds

# tune/eval is component-safe; OOF inside tune is component-safe too
tune=group_mask(h.component.to_numpy(),.50,2026); evaluation=~tune; groups=np.unique(h.loc[tune,"component"]); rng=np.random.RandomState(2027); fold_map={g:i%5 for i,g in enumerate(rng.permutation(groups))}; fold=np.full(len(h),-1,np.int8); fold[tune]=[fold_map[x] for x in h.loc[tune,"component"]]
candidate_rows=[]; oof_store={}
for cfg in META_CONFIGS:
    op=CACHE/f"oof_{cfg['name']}.npy"
    if op.exists():oof=np.load(op)
    else:
        oof=np.full(len(h),np.nan,np.float32)
        for f in range(5):
            tr=np.flatnonzero(tune&(fold!=f)); va=np.flatnonzero(tune&(fold==f)); oof[va]=fit_candidate(cfg,tr,[Mh.iloc[va]],cfg["depth"]*100+f)[0]; print(cfg["name"],"fold",f,"done")
        np.save(op,oof)
    oof_store[cfg["name"]]=oof; alone=macro(yh,oof,ch,tune)
    for a in np.arange(0,.401,.025):
        p=blend(baseh,oof,ch,float(a)); gains=[]
        for f in range(5):m=tune&(fold==f);gains.append(macro(yh,p,ch,m)-macro(yh,baseh,ch,m))
        candidate_rows.append({"candidate":cfg["name"],"alpha":float(a),"tune_oof":macro(yh,p,ch,tune),"candidate_alone":alone,"gain":macro(yh,p,ch,tune)-macro(yh,baseh,ch,tune),"positive_folds":sum(x>0 for x in gains),"worst_fold":min(gains)})
grid=pd.DataFrame(candidate_rows).sort_values(["tune_oof","positive_folds"],ascending=False); stable=grid[(grid.positive_folds>=3)&(grid.gain>=.0003)&(grid.worst_fold>=-.002)]
best=(stable.iloc[0] if len(stable) else grid.iloc[0]); selected_name=str(best.candidate); alpha=float(best.alpha); selected_cfg=next(x for x in META_CONFIGS if x["name"]==selected_name)
print("\nTOP META:\n",grid.head(20).to_string(index=False));print("SELECTED:",selected_name,"alpha",alpha)

# honest independent eval + exact LLM; then production refit on all 72,948
tune_idx=np.flatnonzero(tune); eval_idx=np.flatnonzero(evaluation); eval_dir=MODELS/"eval_selected";eval_dir.mkdir(exist_ok=True)
eval_pred,llm_pred=fit_candidate(selected_cfg,tune_idx,[Mh.iloc[eval_idx],Ml],999,eval_dir)
eval_blend=blend(baseh[eval_idx],eval_pred,ch[eval_idx],alpha); llm_blend=blend(basel,llm_pred,cl,alpha); oof_blend=blend(baseh,oof_store[selected_name],ch,alpha)
honest=np.empty(len(h));honest[tune]=oof_blend[tune];honest[evaluation]=eval_blend
metrics={
 "historical":{"best_public_lb":.5290960422265529,"v53_category_public_lb":.5285395679892638,"e5_v52_manual_reference":.7701124939243256,"old_selective_manual_reference":.771457},
 "manual":{"e5_full":macro(yh,e5h,ch),"old_selective_full":macro(yh,baseh,ch),"old_selective_eval":macro(yh,baseh,ch,evaluation),"meta_eval":macro(yh,eval_blend,ch,evaluation),"meta_honest_full":macro(yh,honest,ch),"eval_gain":macro(yh,eval_blend,ch,evaluation)-macro(yh,baseh,ch,evaluation)},
 "llm":{"e5":macro(yl,e5l,cl),"bge":macro(yl,bgel,cl),"old_selective":macro(yl,basel,cl),"meta":macro(yl,llm_blend,cl),"meta_gain":macro(yl,llm_blend,cl)-macro(yl,basel,cl)},
 "selection":{"candidate":selected_name,"alpha":alpha,"tune_oof":float(best.tune_oof),"positive_folds":int(best.positive_folds),"worst_fold":float(best.worst_fold),"teachers":top_llm}}
metrics["recommended"]=bool(metrics["manual"]["eval_gain"]>.0005 and metrics["llm"]["meta_gain"]>-.001)

prod_dir=MODELS/"production_meta";prod_dir.mkdir(exist_ok=True);fit_candidate(selected_cfg,np.arange(len(h)),[],2028,prod_dir)
for n in top_llm:shutil.copy2(MODELS/f"{n}.cbm",EXPORT/f"{n}.cbm");shutil.copy2(MODELS/f"{n}.json",EXPORT/f"{n}.json")
if (prod_dir/"meta_global.cbm").exists():shutil.copy2(prod_dir/"meta_global.cbm",EXPORT/"meta_global.cbm")
else:
    dst=EXPORT/"meta_categories";dst.mkdir(exist_ok=True)
    for p in prod_dir.glob("*.cbm"):shutil.copy2(p,dst/p.name)
grid.to_csv(EXPORT/"meta_search.csv",index=False);llm_table.to_csv(EXPORT/"llm_bank_metrics.csv",index=False)
percat=[]
for c in np.unique(ch[evaluation]):
    m=evaluation&(ch==c);percat.append({"category":c,"pairs":int(m.sum()),"old_selective":average_precision_score(yh[m],baseh[m]),"meta":average_precision_score(yh[m],honest[m]),"gain":average_precision_score(yh[m],honest[m])-average_precision_score(yh[m],baseh[m])})
pd.DataFrame(percat).sort_values("gain",ascending=False).to_csv(EXPORT/"manual_eval_by_category.csv",index=False)
(EXPORT/"metrics.json").write_text(json.dumps(metrics,ensure_ascii=False,indent=2));(EXPORT/"ensemble_config.json").write_text(json.dumps({"base":"E5 V5.2 + BGE top10 weight .20","meta_candidate":selected_cfg,"meta_alpha":alpha,"manual_features":MANUAL,"meta_features":FEATURES,"llm_teachers":top_llm},ensure_ascii=False,indent=2))
archive=shutil.make_archive("/kaggle/working/catboost_residual_bank_results","zip",EXPORT)
print("\nFINAL METRICS:\n",json.dumps(metrics,ensure_ascii=False,indent=2));print("\nVERDICT:","BUILD SUBMISSION" if metrics["recommended"] else "DO NOT SUBMIT");print("saved:",archive,"elapsed hours:",(time.time()-START)/3600)


INPUTS:
  human: /kaggle/input/datasets/mihailivanovvvv/hakaton-ozon-math-items/matches.parquet
  items_human: /kaggle/input/datasets/mihailivanovvvv/hakaton-ozon-math-items/items_human.parquet
  llm_full: /kaggle/input/datasets/mihailivanovvvv/hakaton-ozon-math-items/matches_llm.parquet
  items_full: /kaggle/input/datasets/mihailivanovvvv/hakaton-ozon-math-items/items.parquet
  polygon_2m: /kaggle/input/datasets/w4stdd/2m-parquet/llm_sample_2m.parquet
GPU: ['Tesla T4', 'Tesla T4']
E5 V5.2: /kaggle/input/models/danilzhukovv/ecup-product-matching-e5/pytorch/default/1 
BGE: /kaggle/input/models/danilzhukovv/ecup-product-matching-bge/pytorch/default/1
human untouched: 72948
exact LLM holdout: 191,555, 1.0 min
LLM soft train: 2000000
profiles 2,588,017/2,588,017
human features 50,000/72,948 7200/s
human features 72,948/72,948 7062/s
LLM holdout features 50,000/191,555 7272/s
LLM holdout features 100,000/191,555 6959/s
LLM holdout features 150,000/191,555 6931/s
LLM holdout features 191,555

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

E5 24/264,503 13/s
E5 24,024/264,503 133/s
E5 48,024/264,503 126/s
E5 72,024/264,503 121/s
E5 96,024/264,503 116/s
E5 120,024/264,503 111/s
E5 144,024/264,503 107/s
E5 168,024/264,503 104/s
E5 192,024/264,503 101/s
E5 216,024/264,503 99/s
E5 240,024/264,503 97/s
E5 264,024/264,503 96/s


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

BGE 512/264,503 346/s
BGE 20,992/264,503 163/s
BGE 41,472/264,503 133/s
BGE 61,952/264,503 115/s
BGE 82,432/264,503 102/s
BGE 102,912/264,503 92/s
BGE 123,392/264,503 86/s
BGE 143,872/264,503 82/s
BGE 164,352/264,503 79/s
BGE 184,832/264,503 77/s
BGE 205,312/264,503 75/s
BGE 225,792/264,503 73/s
BGE 246,272/264,503 72/s

BASELINES manual full: {'E5_V5.2': 0.7701344436902106, 'BGE': 0.7035273889698249, 'old_selective': 0.7714449371925742}
BASELINES exact LLM: {'E5_V5.2': 0.8166039584246516, 'BGE': 0.859740022180717, 'old_selective': 0.8216720423162187}
0:	learn: 0.6770781	test: 0.6770091	best: 0.6770091 (0)	total: 237ms	remaining: 15m 48s
250:	learn: 0.4504734	test: 0.4489086	best: 0.4489086 (250)	total: 20.7s	remaining: 5m 9s
500:	learn: 0.4441640	test: 0.4427702	best: 0.4427702 (500)	total: 40.9s	remaining: 4m 45s
750:	learn: 0.4409527	test: 0.4397977	best: 0.4397977 (750)	total: 1m 1s	remaining: 4m 24s
1000:	learn: 0.4387785	test: 0.4378737	best: 0.4378737 (1000)	total: 1m 21s	remain

IndexError: boolean index did not match indexed array along axis 0; size of axis is 36475 but size of corresponding boolean axis is 72948

In [3]:
from pathlib import Path
p=Path("/kaggle/working/catboost_residual_bank")
print("cache exists:",p.exists())
print([str(x.relative_to(p)) for x in p.rglob("*") if x.is_file()][:30] if p.exists() else [])

cache exists: True
['cache/human_untouched.parquet']


In [4]:
import contextlib, gc, json, os, re, shutil, time, warnings
from difflib import SequenceMatcher
from pathlib import Path

os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
os.environ["TOKENIZERS_PARALLELISM"]="false"; warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import polars as pl
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
import torch
import torch.nn as nn
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import average_precision_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer, PreTrainedTokenizerFast

START=time.time(); ROOT=Path("/kaggle/input")
BASE=Path("/kaggle/input/datasets/mihailivanovvvv/hakaton-ozon-math-items")
def locate_file(expected,name,prefer=""):
    if expected.exists():return expected
    hits=list(ROOT.rglob(name))
    if prefer:hits.sort(key=lambda p:(prefer.lower() not in str(p).lower(),len(str(p))))
    else:hits.sort(key=lambda p:len(str(p)))
    if not hits:raise FileNotFoundError(f"Не найден {name} в /kaggle/input. Подключи нужный Dataset.")
    return hits[0]
HUMAN=locate_file(BASE/"matches.parquet","matches.parquet","hakaton-ozon-math-items")
ITEMS_H=locate_file(BASE/"items_human.parquet","items_human.parquet","hakaton-ozon-math-items")
LLM_FULL=locate_file(BASE/"matches_llm.parquet","matches_llm.parquet","hakaton-ozon-math-items")
ITEMS=locate_file(BASE/"items.parquet","items.parquet","hakaton-ozon-math-items")
POLYGON=locate_file(Path("/kaggle/input/datasets/w4stdd/2m-parquet/llm_sample_2m.parquet"),"llm_sample_2m.parquet","2m-parquet")
WORK=Path("/kaggle/working/catboost_residual_bank"); WORK.mkdir(exist_ok=True)
CACHE=WORK/"cache"; CACHE.mkdir(exist_ok=True); MODELS=WORK/"models"; MODELS.mkdir(exist_ok=True)
EXPORT=WORK/"export"; EXPORT.mkdir(exist_ok=True)
DEV=torch.device("cuda:0"); GPUS=list(range(torch.cuda.device_count())); DEVICES="0:1" if len(GPUS)>1 else "0"
assert len(GPUS)==2,f"Нужно 2 GPU, сейчас видно: {len(GPUS)}"
print("INPUTS:")
for label,path in {"human":HUMAN,"items_human":ITEMS_H,"llm_full":LLM_FULL,"items_full":ITEMS,"polygon_2m":POLYGON}.items():print(f"  {label}: {path}")

E5_FIXED=Path("/kaggle/input/models/danilzhukovv/ecup-product-matching-e5/pytorch/default/1")
BGE_FIXED=Path("/kaggle/input/models/danilzhukovv/ecup-product-matching-bge/pytorch/default/1")
def find_e5():
    if (E5_FIXED/"hybrid_heads.pt").exists(): return E5_FIXED
    hits=[p.parent for p in ROOT.rglob("hybrid_heads.pt") if "v5-3" not in str(p).lower() and "v5_3" not in str(p).lower()]
    if not hits: raise FileNotFoundError("Не найдена E5 V5.2 с hybrid_heads.pt")
    return sorted(hits,key=lambda p:len(str(p)))[0]
def find_bge():
    if (BGE_FIXED/"model.safetensors").exists(): return BGE_FIXED
    for p in ROOT.rglob("config.json"):
        try:
            c=json.loads(p.read_text())
            if c.get("hidden_size")==1024 and c.get("num_hidden_layers")==24 and (p.parent/"model.safetensors").exists(): return p.parent
        except: pass
    raise FileNotFoundError("Не найдена BGE-M3")
E5_DIR,BGE_DIR=find_e5(),find_bge()
print("GPU:",[torch.cuda.get_device_name(i) for i in GPUS]); print("E5 V5.2:",E5_DIR,"\nBGE:",BGE_DIR)

# ---------- component-safe datasets ----------
def components(a,b):
    parent={}
    def find(x):
        parent.setdefault(x,x)
        while parent[x]!=x: parent[x]=parent[parent[x]]; x=parent[x]
        return x
    for x,y in zip(a,b):
        x,y=find(x),find(y)
        if x!=y: parent[y]=x
    return np.fromiter((find(x) for x in a),np.int64,len(a))
def group_mask(comp,frac,seed):
    groups=np.unique(comp); rng=np.random.RandomState(seed); chosen=set(groups[rng.rand(len(groups))<frac])
    return np.fromiter((x in chosen for x in comp),bool,len(comp))

HP=CACHE/"human_untouched.parquet"; LP=CACHE/"llm_holdout.parquet"; TP=CACHE/"llm_train_2m.parquet"
if not HP.exists():
    h=pl.read_parquet(HUMAN,columns=["id1","id2","target"]).to_pandas(); comp=components(h.id1.to_numpy(),h.id2.to_numpy()); m=group_mask(comp,.20,42)
    h=h.loc[m].copy(); h["component"]=comp[m]; h.to_parquet(HP,index=False); print("human untouched:",len(h))
if not LP.exists():
    t=time.time(); llm=pl.read_parquet(LLM_FULL,columns=["id1","id2","target"]).to_pandas(); comp=components(llm.id1.to_numpy(),llm.id2.to_numpy()); m=group_mask(comp,.03,13)
    llm=llm.loc[m].copy(); val_comp=comp[m]; clean=((llm.target<=.2)|(llm.target>=.8)).to_numpy(); llm=llm.loc[clean].copy(); llm["target_soft"]=llm.target; llm["target"]=(llm.target>=.5).astype(np.int8); llm["component"]=val_comp[clean]
    llm.to_parquet(LP,index=False); print(f"exact LLM holdout: {len(llm):,}, {(time.time()-t)/60:.1f} min")
if not TP.exists():
    hold=pl.read_parquet(LP,columns=["id1","id2"]).with_columns(pl.min_horizontal("id1","id2").alias("_a"),pl.max_horizontal("id1","id2").alias("_b")).select("_a","_b").unique()
    train=(pl.read_parquet(POLYGON,columns=["id1","id2","target"])
           .with_columns(pl.min_horizontal("id1","id2").alias("_a"),pl.max_horizontal("id1","id2").alias("_b"))
           .join(hold,on=["_a","_b"],how="anti").drop("_a","_b"))
    if len(train)>2_000_000: train=train.sample(n=2_000_000,seed=314159,shuffle=True)
    train.write_parquet(TP,compression="zstd"); print("LLM soft train:",len(train))
h=pd.read_parquet(HP); lh=pd.read_parquet(LP); lt=pd.read_parquet(TP)
assert len(h)==72_948 and len(lh)==191_555

# ---------- exact V5 structured features ----------
SPACE_RE=re.compile(r"\s+"); MULTIPLY_RE=re.compile(r"[×хХ*]"); TOKEN_RE=re.compile(r"[0-9a-zа-я]+",re.I)
SIZE_RE=re.compile(r"(?:размер|р-р|size)\s*[:=\-]?\s*([0-9]{2,3}(?:[./-][0-9]{1,3})?|xxxs|xxs|xs|s|m|l|xl|xxl|xxxl)",re.I)
UNIT_RE=re.compile(r"(\d+(?:[.,]\d+)?)\s*(мл|ml|л|l|мг|mg|г|gr|g|кг|kg|мм|mm|см|cm|м|m|вт|w|квт|kw|шт|pcs|pack)\b",re.I)
PACK_RE=re.compile(r"(\d+)\s*[xх×*]\s*(\d+(?:[.,]\d+)?)\s*(мл|ml|л|l|мг|mg|г|gr|g|кг|kg|шт|pcs)",re.I)
KEY_ORDER=["бренд","brand","производитель","артикул","партномер","part number","partnumber","oem","код","sku","модель","model","размер","size","рост","обхват","пол","gender","цвет","color","материал","material","объем","обьем","volume","вес","weight","количество","комплектация","упаков","тип","type"]
ALIASES={"brand":["бренд","brand","производитель"],"article":["артикул","sku","oem","партномер","part number","partnumber","код производителя"],"model":["модель","model"],"size":["размер производителя","размер обуви","размер одежды","размер","size","рост"],"color":["основной цвет","цвет","color","расцветка"],"material":["материал верха","материал","material"],"gender":["пол","gender"]}
HOMO=str.maketrans({"а":"a","в":"b","с":"c","е":"e","н":"h","к":"k","м":"m","о":"o","р":"p","т":"t","х":"x","у":"y"})
COLORS={x.replace("ё","е"):y for x,y in {"черный":"черный","чёрный":"черный","белый":"белый","серый":"серый","серебристый":"серебристый","красный":"красный","бордовый":"бордовый","синий":"синий","голубой":"голубой","зеленый":"зеленый","зелёный":"зеленый","желтый":"желтый","жёлтый":"желтый","оранжевый":"оранжевый","розовый":"розовый","фиолетовый":"фиолетовый","бежевый":"бежевый","коричневый":"коричневый","золотой":"золотой","золотистый":"золотой"}.items()}
def norm(x): return "" if x is None else SPACE_RE.sub(" ",MULTIPLY_RE.sub("x",str(x).lower().replace("ё","е").replace(",","."))).strip()
def attrs_dict(x):
    if isinstance(x,dict): d=x
    elif isinstance(x,str):
        try:d=json.loads(x)
        except:d={}
    else:d={}
    return {norm(k):norm(v) for k,v in d.items() if norm(k) and norm(v)} if isinstance(d,dict) else {}
def pick(d,a):
    for alias in a:
        for k,v in d.items():
            if alias in k:return v
    return ""
def compact(x,limit=96):return SPACE_RE.sub(" ",re.sub(r"[^0-9a-zа-я.+/_\- ]+"," ",norm(x))).strip()[:limit]
def code_norm(x):return re.sub(r"[^0-9a-z]+","",compact(x).translate(HOMO))
def size_norm(x):return re.sub(r"\s*/\s*","/",re.sub(r"\s*-\s*","-",compact(x,64).replace("–","-").replace("—","-")))
def color_norm(x):
    text=compact(x,64); found=[v for k,v in COLORS.items() if k in text]; return "/".join(sorted(set(found))) if found else text
def fallback_color(x):
    text=norm(x); return "/".join(sorted(set(v for k,v in COLORS.items() if k in text)))
def quantity_tokens(text):
    def cv(v,u):
        u=u.lower()
        if u in {"мл","ml"}:return "volume_ml",v
        if u in {"л","l"}:return "volume_ml",v*1000
        if u in {"мг","mg"}:return "mass_g",v/1000
        if u in {"г","gr","g"}:return "mass_g",v
        if u in {"кг","kg"}:return "mass_g",v*1000
        if u in {"мм","mm"}:return "length_mm",v
        if u in {"см","cm"}:return "length_mm",v*10
        if u in {"м","m"}:return "length_mm",v*1000
        if u in {"вт","w"}:return "power_w",v
        if u in {"квт","kw"}:return "power_w",v*1000
        return "count",v
    text=norm(text); out=set()
    for c,a,u in PACK_RE.findall(text): kind,v=cv(float(a.replace(",",".")),u); out|={f"{kind}:{round(float(c)*v,4)}",f"pack_count:{round(float(c),4)}"}
    for a,u in UNIT_RE.findall(text): kind,v=cv(float(a.replace(",",".")),u); out.add(f"{kind}:{round(v,4)}")
    return frozenset(out)
class Item:
    __slots__=("name","tokens","brand","article","model","size","color","material","gender","quantity","category")
    def __init__(self,name,attributes,category):
        d=attrs_dict(attributes); self.name=norm(name); self.tokens=frozenset(x for x in TOKEN_RE.findall(self.name) if len(x)>=2); self.brand=compact(pick(d,ALIASES["brand"])); self.article=code_norm(pick(d,ALIASES["article"])); self.model=code_norm(pick(d,ALIASES["model"])); m=SIZE_RE.search(self.name); self.size=size_norm(pick(d,ALIASES["size"])) or (size_norm(m.group(1)) if m else ""); self.color=color_norm(pick(d,ALIASES["color"])) or fallback_color(self.name); self.material=compact(pick(d,ALIASES["material"])); self.gender=compact(pick(d,ALIASES["gender"])); self.category=str(category); self.quantity=quantity_tokens(self.name+" "+" ".join(d.values())[:1800])
def jaccard(a,b):return len(a&b)/len(a|b) if a|b else 0.
def contain(a,b):return len(a&b)/max(1,min(len(a),len(b))) if a and b else 0.
def eq(a,b):return -1. if not a or not b else float(a==b)
def structured(a,b):
    n1,n2=a.name,b.name; v=[float(bool(n1) and n1==n2),float(bool(n1) and bool(n2) and (n1 in n2 or n2 in n1)),SequenceMatcher(None,n1,n2).ratio(),jaccard(a.tokens,b.tokens),contain(a.tokens,b.tokens),min(len(n1),len(n2))/max(1,max(len(n1),len(n2))),min(abs(len(n1)-len(n2))/100,5)]
    for x,y in [(a.brand,b.brand),(a.article,b.article),(a.model,b.model),(a.size,b.size),(a.color,b.color),(a.material,b.material),(a.gender,b.gender)]:v += [eq(x,y),float(bool(x) and bool(y))]
    v += [jaccard(a.quantity,b.quantity),float(bool(a.quantity) and bool(b.quantity) and not a.quantity&b.quantity),float(bool(a.quantity) and bool(b.quantity)),SequenceMatcher(None,a.article,b.article).ratio() if a.article and b.article else 0.,SequenceMatcher(None,a.model,b.model).ratio() if a.model and b.model else 0.]
    return np.asarray(v,np.float32)
MANUAL=[f"structured_{i:02d}" for i in range(26)]

def e5_text(name,attributes,category):
    d=attrs_dict(attributes); used=set(); priority=[]
    for wanted in KEY_ORDER:
        for k,v in d.items():
            if k not in used and wanted in k:priority.append(f"{k}: {v}"); used.add(k)
    attrs=" ; ".join(priority+[f"{k}: {v}" for k,v in d.items() if k not in used])[:620]
    return f"категория: {norm(category)} | название: {norm(name)} | атрибуты: {attrs}"
BGE_KEYS=["бренд","артикул","партномер","oem","код","модель","размер","цвет","объем","обьем","вес","тип","материал","количество"]
CYR2LAT=str.maketrans("аеорсухАЕОРСУХКМТВНЗЅІі","aeopcyxAEOPCYXKMTBH3SIi"); LAT2CYR=str.maketrans("aeopcyxAEOPCYX","аеорсухАЕОРСУХ")
BUNIT_RE=re.compile(r"(\d+(?:[.,]\d+)?)\s*(мл|ml|л|l|мг|mg|г|g|гр|кг|kg|мм|mm|см|cm|м|мб|mb|гб|gb|тб|tb|вт|w|квт|kw|мач|mah)\b",re.I)
BMUL={"мл":("ml",1),"ml":("ml",1),"л":("ml",1000),"l":("ml",1000),"мг":("g",.001),"mg":("g",.001),"г":("g",1),"g":("g",1),"гр":("g",1),"кг":("g",1000),"kg":("g",1000),"мм":("mm",1),"mm":("mm",1),"см":("mm",10),"cm":("mm",10),"м":("mm",1000),"мб":("gb",.001),"mb":("gb",.001),"гб":("gb",1),"gb":("gb",1),"тб":("gb",1024),"tb":("gb",1024),"вт":("w",1),"w":("w",1),"квт":("w",1000),"kw":("w",1000),"мач":("mah",1),"mah":("mah",1)}
QRES=[re.compile(r"(\d+)\s*шт"),re.compile(r"набор\w*\s+из\s+(\d+)"),re.compile(r"(\d+)\s*(?:набор|упаков|комплект)\w*\s+по\s+(\d+)"),re.compile(r"[xх*](\d+)\b")]
def bge_text(name,attributes):
    parts=[str(name) if name is not None else ""]
    try:d=json.loads(attributes) if isinstance(attributes,str) else {}
    except:d={}
    if isinstance(d,dict) and d:
        low={str(k).lower():str(v) for k,v in d.items() if v}; picked=[]; used=set()
        for wanted in BGE_KEYS:
            for k,v in low.items():
                if wanted in k and k not in used:picked.append(f"{k}:{v}"); used.add(k)
        parts.append(" ; ".join(picked+[f"{k}:{v}" for k,v in low.items() if k not in used])[:520])
    base=" | ".join(parts).replace("ё","е").replace("Ё","Е"); fixed=[]
    for token in base.split():
        c=sum("\u0400"<=x<="\u04ff" for x in token); l=sum(x.isascii() and x.isalpha() for x in token); fixed.append(token.translate(CYR2LAT if l>=c else LAT2CYR) if c and l else token)
    base=re.sub(r"[×хХ](?=\d)","x"," ".join(fixed)); extras=[]; units=set()
    for m in BUNIT_RE.finditer(base):u,k=BMUL[m.group(2).lower()];units.add(f"{float(m.group(1).replace(',','.'))*k:g}{u}")
    if units:extras.append("ед: "+" ".join(sorted(units)[:12]))
    q=None;m=QRES[2].search(base.lower())
    if m:q=int(m.group(1))*int(m.group(2))
    else:
        for r in (QRES[0],QRES[1],QRES[3]):
            m=r.search(base.lower())
            if m:q=int(m.group(1));break
    if q and 1<q<=1000:extras.append(f"кол-во: {q}")
    return (base+(" | "+" | ".join(extras) if extras else ""))[:2000]

# ---------- profiles + feature cache ----------
HF=CACHE/"human_features.npy"; LF=CACHE/"llm_holdout_features.npy"; TF=CACHE/"llm_train_features.npy"
HC=CACHE/"human_categories.npy"; LC=CACHE/"llm_holdout_categories.npy"; TC=CACHE/"llm_train_categories.npy"; TEXTS=CACHE/"score_texts.parquet"
if not all(p.exists() for p in [HF,LF,TF,HC,LC,TC,TEXTS]):
    need=set(h.id1)|set(h.id2)|set(lh.id1)|set(lh.id2)|set(lt.id1)|set(lt.id2); score_need=set(h.id1)|set(h.id2)|set(lh.id1)|set(lh.id2)
    profiles={}; text_rows=[]; remaining=set(need); t=time.time()
    for item_path in [ITEMS,ITEMS_H]:
        if not remaining:break
        pf=pq.ParquetFile(item_path); values=pa.array(list(remaining),type=pf.schema_arrow.field("id").type)
        for batch in pf.iter_batches(columns=["id","name","attributes","category"],batch_size=300_000,use_threads=True):
            ids=batch.column(batch.schema.get_field_index("id")); selected=batch.filter(pc.is_in(ids,value_set=values))
            for i,n,a,c in selected.to_pandas().itertuples(index=False,name=None):
                if i in remaining:
                    profiles[i]=Item(n,a,c); remaining.remove(i)
                    if i in score_need:text_rows.append((i,e5_text(n,a,c),bge_text(n,a),str(c)))
        print(f"profiles {len(profiles):,}/{len(need):,}")
    assert not remaining,f"Missing items: {len(remaining)}"
    pd.DataFrame(text_rows,columns=["id","e5_text","bge_text","category"]).to_parquet(TEXTS,index=False)
    def build(df,fpath,cpath,label):
        x=np.lib.format.open_memmap(fpath,mode="w+",dtype=np.float32,shape=(len(df),26)); cats=np.empty(len(df),dtype="<U64"); started=time.time(); a=df.id1.to_numpy(); b=df.id2.to_numpy()
        for s in range(0,len(df),50_000):
            e=min(s+50_000,len(df))
            for j in range(s,e):x[j]=structured(profiles[a[j]],profiles[b[j]]); cats[j]=profiles[a[j]].category
            print(f"{label} {e:,}/{len(df):,} {e/max(time.time()-started,1):.0f}/s")
        x.flush(); np.save(cpath,cats)
    if not HF.exists():build(h,HF,HC,"human features")
    if not LF.exists():build(lh,LF,LC,"LLM holdout features")
    if not TF.exists():build(lt,TF,TC,"LLM train features")
    del profiles,text_rows; gc.collect(); print(f"feature stage {(time.time()-t)/60:.1f} min")
Xh=np.load(HF,mmap_mode="r"); Xl=np.load(LF,mmap_mode="r"); Xt=np.load(TF,mmap_mode="r"); ch=np.load(HC); cl=np.load(LC); ct=np.load(TC)

# ---------- E5 V5.2 + BGE scores on honest holdouts ----------
EH=CACHE/"e5_human.npy"; EL=CACHE/"e5_llm.npy"; BH=CACHE/"bge_human.npy"; BL=CACHE/"bge_llm.npy"
texts_df=pd.read_parquet(TEXTS); e5map=dict(zip(texts_df.id,texts_df.e5_text)); bgemap=dict(zip(texts_df.id,texts_df.bge_text))
pair_all=pd.concat([h[["id1","id2"]],lh[["id1","id2"]]],ignore_index=True); cats_all=np.concatenate([ch,cl]); struct_all=np.concatenate([np.asarray(Xh),np.asarray(Xl)]); split=len(h)
class HybridStageB(nn.Module):
    def __init__(self,base,ncat,nstruct):
        super().__init__(); self.base=base; self.encoder=base.roberta if hasattr(base,"roberta") else base.xlm_roberta; z=int(base.config.hidden_size); self.category_embedding=nn.Embedding(ncat,24); self.category_residual=nn.Sequential(nn.Linear(z+24,128),nn.GELU(),nn.Dropout(.1),nn.Linear(128,1)); self.structured_residual=nn.Sequential(nn.LayerNorm(nstruct),nn.Linear(nstruct,64),nn.GELU(),nn.Dropout(.1),nn.Linear(64,1)); self.auxiliary_head=nn.Sequential(nn.Linear(z,128),nn.GELU(),nn.Dropout(.1),nn.Linear(128,6))
    def forward(self,input_ids,attention_mask=None,token_type_ids=None,category_ids=None,structured=None,**kw):
        z={"input_ids":input_ids,"attention_mask":attention_mask,"return_dict":True}
        if token_type_ids is not None:z["token_type_ids"]=token_type_ids
        seq=self.encoder(**z).last_hidden_state; pooled=seq[:,0]; return self.base.classifier(seq).view(-1)+self.category_residual(torch.cat([pooled,self.category_embedding(category_ids)],-1)).view(-1)+self.structured_residual(structured.float()).view(-1),self.auxiliary_head(pooled)
@torch.inference_mode()
def predict_e5():
    if EH.exists() and EL.exists():return np.load(EH),np.load(EL)
    head=torch.load(E5_DIR/"hybrid_heads.pt",map_location="cpu",weights_only=False); categories=[str(x) for x in head["categories"]]; cmap={c:i for i,c in enumerate(categories)}; assert head.get("pair_mode")=="plain_v2"
    base=AutoModelForSequenceClassification.from_pretrained(E5_DIR,local_files_only=True,dtype=torch.float32); core=HybridStageB(base,len(categories),int(head["n_structured_features"])); inc=core.load_state_dict(head["state_dict"],strict=False); assert not inc.unexpected_keys; core.to(DEV).eval(); model=nn.DataParallel(core,device_ids=GPUS)
    try:tok=AutoTokenizer.from_pretrained(E5_DIR,local_files_only=True,use_fast=True)
    except:tok=PreTrainedTokenizerFast(tokenizer_file=str(E5_DIR/"tokenizer.json"),bos_token="<s>",cls_token="<s>",eos_token="</s>",sep_token="</s>",unk_token="<unk>",pad_token="<pad>",mask_token="<mask>",model_max_length=512)
    a=pair_all.id1.to_numpy(); b=pair_all.id2.to_numpy(); order=np.argsort([len(e5map[x])+len(e5map[y]) for x,y in zip(a,b)],kind="stable"); sorted_pred=np.empty(len(pair_all),np.float32); bs=24; started=time.time()
    for s in range(0,len(order),bs):
        pos=order[s:s+bs]; ab=[(e5map[a[i]],e5map[b[i]]) for i in pos]; pairs=ab+[(y,x) for x,y in ab]; enc=tok([x for x,y in pairs],[y for x,y in pairs],padding=True,truncation=True,max_length=int(head["max_length"]),return_tensors="pt"); enc={k:v.to(DEV,non_blocking=True) for k,v in enc.items()}; st=np.concatenate([struct_all[pos],struct_all[pos]]); ci=np.asarray([cmap[str(cats_all[i])] for i in pos]*2,np.int64)
        with torch.autocast("cuda",dtype=torch.float16):logits,_=model(**enc,structured=torch.from_numpy(st).to(DEV),category_ids=torch.from_numpy(ci).to(DEV))
        n=len(pos); p=torch.sigmoid(logits.float()); sorted_pred[s:s+n]=((p[:n]+p[n:])/2).cpu().numpy()
        if s//bs%1000==0:print(f"E5 {s+n:,}/{len(order):,} {(s+n)/max(time.time()-started,1):.0f}/s")
    pred=np.empty(len(order),np.float32); pred[order]=sorted_pred; np.save(EH,pred[:split]); np.save(EL,pred[split:]); del model,core,base,tok; gc.collect(); torch.cuda.empty_cache(); return pred[:split],pred[split:]
@torch.inference_mode()
def predict_bge():
    if BH.exists() and BL.exists():return np.load(BH),np.load(BL)
    tok=AutoTokenizer.from_pretrained(BGE_DIR,local_files_only=True); core=AutoModelForSequenceClassification.from_pretrained(BGE_DIR,local_files_only=True,dtype=torch.float16).to(DEV).eval(); model=nn.DataParallel(core,device_ids=GPUS); a=pair_all.id1.to_numpy(); b=pair_all.id2.to_numpy(); order=np.argsort([len(bgemap[x])+len(bgemap[y]) for x,y in zip(a,b)],kind="stable"); pred=np.empty(len(pair_all),np.float32); bs=512; started=time.time()
    for s in range(0,len(order),bs):
        pos=order[s:s+bs]; x=[bgemap[a[i]] for i in pos]; y=[bgemap[b[i]] for i in pos]; total=0
        for u,v in ((x,y),(y,x)):
            enc=tok(u,v,padding=True,truncation=True,max_length=320,pad_to_multiple_of=8,return_tensors="pt"); enc={k:v.to(DEV,non_blocking=True) for k,v in enc.items()}
            with torch.autocast("cuda",dtype=torch.float16):total+=torch.sigmoid(model(**enc).logits.squeeze(-1).float()).cpu().numpy()
        pred[pos]=total/2
        if s//bs%40==0:print(f"BGE {s+len(pos):,}/{len(order):,} {(s+len(pos))/max(time.time()-started,1):.0f}/s")
    np.save(BH,pred[:split]); np.save(BL,pred[split:]); del model,core,tok; gc.collect(); torch.cuda.empty_cache(); return pred[:split],pred[split:]
e5h,e5l=predict_e5(); bgeh,bgel=predict_bge(); del pair_all,struct_all,texts_df,e5map,bgemap; gc.collect()

# ---------- metrics/rank helpers ----------
def rank(x):return pd.Series(x).rank(method="average",pct=True).to_numpy()
def cat_rank(x,c):
    out=np.empty(len(x),np.float64)
    for z in np.unique(c):m=c==z; out[m]=rank(np.asarray(x)[m])
    return out
def macro(y,p,c,m=None):
    if m is None:m=np.ones(len(y),bool)
    return float(np.mean([average_precision_score(y[m&(c==z)],p[m&(c==z)]) for z in np.unique(c[m])]))
def selective(e5,bge,c,frac=.10,w=.20):
    out=cat_rank(e5,c)
    for z in np.unique(c):
        ix=np.flatnonzero(c==z); k=max(1,int(np.ceil(len(ix)*frac))); chosen=ix[np.argsort(e5[ix],kind="stable")[-k:]]; mixed=(1-w)*rank(e5[chosen])+w*rank(bge[chosen]); out[chosen]=(len(ix)-k)/len(ix)+rank(mixed)*k/len(ix)
    return out
def blend(base,extra,c,a):return (1-a)*cat_rank(base,c)+a*cat_rank(extra,c)
yh=h.target.to_numpy(np.int8); yl=lh.target.to_numpy(np.int8); baseh=selective(e5h,bgeh,ch); basel=selective(e5l,bgel,cl)
print("\nBASELINES manual full:",{"E5_V5.2":macro(yh,e5h,ch),"BGE":macro(yh,bgeh,ch),"old_selective":macro(yh,baseh,ch)})
print("BASELINES exact LLM:",{"E5_V5.2":macro(yl,e5l,cl),"BGE":macro(yl,bgel,cl),"old_selective":macro(yl,basel,cl)})

# ---------- 8 LLM-soft CatBoost teachers ----------
LLM_CONFIGS=[
    {"name":"llm_d6_s11","depth":6,"lr":.035,"l2":6,"seed":11},{"name":"llm_d7_s22","depth":7,"lr":.03,"l2":8,"seed":22},
    {"name":"llm_d8_s33","depth":8,"lr":.025,"l2":10,"seed":33},{"name":"llm_d9_s44","depth":9,"lr":.022,"l2":12,"seed":44},
    {"name":"llm_d6_s55","depth":6,"lr":.025,"l2":14,"seed":55},{"name":"llm_d7_s66","depth":7,"lr":.02,"l2":18,"seed":66},
    {"name":"llm_d8_s77","depth":8,"lr":.02,"l2":20,"seed":77},{"name":"llm_d10_s88","depth":10,"lr":.015,"l2":24,"seed":88}]
def frame(x,c):
    d=pd.DataFrame(np.asarray(x),columns=MANUAL,copy=False); d.insert(0,"category",np.asarray(c).astype(str)); return d
Ft=frame(Xt,ct); Fh_manual=frame(Xh,ch); Fl_manual=frame(Xl,cl); yt=lt.target.to_numpy(np.float32)
rng=np.random.RandomState(730); perm=rng.permutation(len(yt)); cut=int(len(yt)*.92); tr,va=perm[:cut],perm[cut:]
counts=pd.Series(ct).value_counts(); med=float(counts.median()); cw={k:float(np.clip((med/v)**.5,.65,1.7)) for k,v in counts.items()}; sw=np.asarray([cw[x] for x in ct],np.float32)*(.65+1.35*np.abs(yt-.5)*2)
train_pool=Pool(Ft.iloc[tr],yt[tr],cat_features=["category"],weight=sw[tr]); val_pool=Pool(Ft.iloc[va],yt[va],cat_features=["category"],weight=sw[va]); full_pool=Pool(Ft,yt,cat_features=["category"],weight=sw)
llm_rows=[]
for cfg in LLM_CONFIGS:
    mp=MODELS/f"{cfg['name']}.cbm"; ph=CACHE/f"{cfg['name']}_human.npy"; plp=CACHE/f"{cfg['name']}_llm.npy"; meta=MODELS/f"{cfg['name']}.json"
    if not (mp.exists() and ph.exists() and plp.exists()):
        p=dict(iterations=4000,depth=cfg["depth"],learning_rate=cfg["lr"],l2_leaf_reg=cfg["l2"],random_strength=1,loss_function="CrossEntropy",eval_metric="CrossEntropy",task_type="GPU",devices=DEVICES,border_count=128,random_seed=cfg["seed"],allow_writing_files=False,verbose=250,od_type="Iter",od_wait=180)
        model=CatBoostClassifier(**p); model.fit(train_pool,eval_set=val_pool,use_best_model=True); best=max(200,model.get_best_iteration()+1); p.pop("od_type");p.pop("od_wait");p["iterations"]=best;p["verbose"]=300
        final=CatBoostClassifier(**p); final.fit(full_pool); final.save_model(mp); np.save(ph,final.predict_proba(Fh_manual)[:,1]); np.save(plp,final.predict_proba(Fl_manual)[:,1]); meta.write_text(json.dumps({**cfg,"best_iteration":best},indent=2)); del model,final;gc.collect();torch.cuda.empty_cache()
    hp=np.load(ph); lp=np.load(plp); llm_rows.append({"name":cfg["name"],"manual":macro(yh,hp,ch),"llm":macro(yl,lp,cl)})
llm_table=pd.DataFrame(llm_rows).sort_values("llm",ascending=False); print("\nLLM SOFT BANK:\n",llm_table.to_string(index=False)); top_llm=llm_table.head(3).name.tolist(); print("top LLM teachers:",top_llm)
del train_pool,val_pool,full_pool,Ft,sw;gc.collect();torch.cuda.empty_cache()

# ---------- residual meta features ----------
def logit(x):x=np.clip(x,1e-5,1-1e-5);return np.log(x/(1-x))
def meta_frame(manual,c,e5,bge,base,domain_preds):
    d=pd.DataFrame(np.asarray(manual),columns=MANUAL,copy=False); d.insert(0,"category",np.asarray(c).astype(str)); d["e5"]=e5;d["bge"]=bge;d["e5_logit"]=logit(e5);d["bge_logit"]=logit(bge);d["e5_rank"]=cat_rank(e5,c);d["bge_rank"]=cat_rank(bge,c);d["old_selective"]=base;d["score_diff"]=bge-e5;d["rank_diff"]=d.bge_rank-d.e5_rank;d["score_absdiff"]=np.abs(bge-e5)
    for name,p in domain_preds.items():d[f"teacher_{name}"]=p
    a=np.column_stack(list(domain_preds.values()));d["teacher_mean"]=a.mean(1);d["teacher_std"]=a.std(1);d["teacher_min"]=a.min(1);d["teacher_max"]=a.max(1);return d
hteach={n:np.load(CACHE/f"{n}_human.npy") for n in top_llm}; lteach={n:np.load(CACHE/f"{n}_llm.npy") for n in top_llm}
Mh=meta_frame(Xh,ch,e5h,bgeh,baseh,hteach); Ml=meta_frame(Xl,cl,e5l,bgel,basel,lteach); FEATURES=[x for x in Mh.columns if x!="category"]

META_CONFIGS=[
 {"name":"global_d6","kind":"global","depth":6,"iterations":1400,"lr":.025,"l2":8,"hard":False},
 {"name":"global_d7","kind":"global","depth":7,"iterations":1500,"lr":.022,"l2":10,"hard":False},
 {"name":"global_d8","kind":"global","depth":8,"iterations":1600,"lr":.018,"l2":14,"hard":False},
 {"name":"global_d9","kind":"global","depth":9,"iterations":1400,"lr":.018,"l2":18,"hard":False},
 {"name":"global_hard_d7","kind":"global","depth":7,"iterations":1700,"lr":.02,"l2":12,"hard":True},
 {"name":"global_hard_d8","kind":"global","depth":8,"iterations":1600,"lr":.018,"l2":18,"hard":True},
 {"name":"category_d5","kind":"category","depth":5,"iterations":900,"lr":.025,"l2":12,"hard":False},
 {"name":"category_hard_d6","kind":"category","depth":6,"iterations":1100,"lr":.02,"l2":16,"hard":True}]
counts=pd.Series(ch).value_counts(); med=float(counts.median()); category_weight={k:float(np.clip((med/v)**.5,.7,1.5)) for k,v in counts.items()}
def weights(indices,hard):
    w=np.asarray([category_weight[x] for x in ch[indices]],np.float32)
    if hard:
        yy=yh[indices]; ss=e5h[indices]; w*=1+3*((yy==0)&(ss>.55))+1.5*((yy==1)&(ss<.45))
    return w
def fit_candidate(cfg,train_idx,out_frames,seed,save_dir=None):
    preds=[np.empty(len(x),np.float64) for x in out_frames]
    if cfg["kind"]=="global":
        p=dict(iterations=cfg["iterations"],depth=cfg["depth"],learning_rate=cfg["lr"],l2_leaf_reg=cfg["l2"],random_strength=1,loss_function="Logloss",task_type="GPU",devices=DEVICES,border_count=128,random_seed=seed,allow_writing_files=False,verbose=False)
        model=CatBoostClassifier(**p); model.fit(Pool(Mh.iloc[train_idx],yh[train_idx],cat_features=["category"],weight=weights(train_idx,cfg["hard"])))
        for i,x in enumerate(out_frames):preds[i]=model.predict_proba(x)[:,1]
        if save_dir:model.save_model(Path(save_dir)/"meta_global.cbm")
        del model
    else:
        Path(save_dir).mkdir(exist_ok=True,parents=True) if save_dir else None
        for cat in np.unique(ch):
            tr=train_idx[ch[train_idx]==cat]
            if len(tr)<100 or len(np.unique(yh[tr]))<2:raise RuntimeError(f"Too little category train: {cat}")
            p=dict(iterations=cfg["iterations"],depth=cfg["depth"],learning_rate=cfg["lr"],l2_leaf_reg=cfg["l2"],random_strength=1,loss_function="Logloss",task_type="CPU",thread_count=4,random_seed=seed,allow_writing_files=False,verbose=False)
            model=CatBoostClassifier(**p); model.fit(Mh.iloc[tr][FEATURES],yh[tr],sample_weight=weights(tr,cfg["hard"]));
            for i,x in enumerate(out_frames):
                m=x.category.to_numpy()==cat
                if m.any():preds[i][m]=model.predict_proba(x.loc[m,FEATURES])[:,1]
            if save_dir:model.save_model(Path(save_dir)/(re.sub(r"[^0-9a-zа-я]+","_",str(cat).lower()).strip("_")+".cbm"))
            del model
    gc.collect();torch.cuda.empty_cache();return preds

# tune/eval is component-safe; OOF inside tune is component-safe too
tune=group_mask(h.component.to_numpy(),.50,2026); evaluation=~tune; groups=np.unique(h.loc[tune,"component"]); rng=np.random.RandomState(2027); fold_map={g:i%5 for i,g in enumerate(rng.permutation(groups))}; fold=np.full(len(h),-1,np.int8); fold[tune]=[fold_map[x] for x in h.loc[tune,"component"]]
candidate_rows=[]; oof_store={}
for cfg in META_CONFIGS:
    op=CACHE/f"oof_{cfg['name']}.npy"
    if op.exists():oof=np.load(op)
    else:
        oof=np.full(len(h),np.nan,np.float32)
        for f in range(5):
            tr=np.flatnonzero(tune&(fold!=f)); va=np.flatnonzero(tune&(fold==f)); oof[va]=fit_candidate(cfg,tr,[Mh.iloc[va]],cfg["depth"]*100+f)[0]; print(cfg["name"],"fold",f,"done")
        np.save(op,oof)
    oof_store[cfg["name"]]=oof; alone=macro(yh,oof,ch,tune)
    for a in np.arange(0,.401,.025):
        p=blend(baseh,oof,ch,float(a)); gains=[]
        for f in range(5):m=tune&(fold==f);gains.append(macro(yh,p,ch,m)-macro(yh,baseh,ch,m))
        candidate_rows.append({"candidate":cfg["name"],"alpha":float(a),"tune_oof":macro(yh,p,ch,tune),"candidate_alone":alone,"gain":macro(yh,p,ch,tune)-macro(yh,baseh,ch,tune),"positive_folds":sum(x>0 for x in gains),"worst_fold":min(gains)})
grid=pd.DataFrame(candidate_rows).sort_values(["tune_oof","positive_folds"],ascending=False); stable=grid[(grid.positive_folds>=3)&(grid.gain>=.0003)&(grid.worst_fold>=-.002)]
best=(stable.iloc[0] if len(stable) else grid.iloc[0]); selected_name=str(best.candidate); alpha=float(best.alpha); selected_cfg=next(x for x in META_CONFIGS if x["name"]==selected_name)
print("\nTOP META:\n",grid.head(20).to_string(index=False));print("SELECTED:",selected_name,"alpha",alpha)

# honest independent eval + exact LLM; then production refit on all 72,948
tune_idx=np.flatnonzero(tune); eval_idx=np.flatnonzero(evaluation); eval_dir=MODELS/"eval_selected";eval_dir.mkdir(exist_ok=True)
eval_pred,llm_pred=fit_candidate(selected_cfg,tune_idx,[Mh.iloc[eval_idx],Ml],999,eval_dir)
eval_blend=blend(baseh[eval_idx],eval_pred,ch[eval_idx],alpha); llm_blend=blend(basel,llm_pred,cl,alpha); oof_blend=blend(baseh,oof_store[selected_name],ch,alpha)
honest=np.empty(len(h));honest[tune]=oof_blend[tune];honest[evaluation]=eval_blend
metrics={
 "historical":{"best_public_lb":.5290960422265529,"v53_category_public_lb":.5285395679892638,"e5_v52_manual_reference":.7701124939243256,"old_selective_manual_reference":.771457},
 "manual":{"e5_full":macro(yh,e5h,ch),"old_selective_full":macro(yh,baseh,ch),"old_selective_eval":macro(yh[eval_idx],baseh[eval_idx],ch[eval_idx]),"meta_eval":macro(yh[eval_idx],eval_blend,ch[eval_idx]),"meta_honest_full":macro(yh,honest,ch),"eval_gain":macro(yh[eval_idx],eval_blend,ch[eval_idx])-macro(yh[eval_idx],baseh[eval_idx],ch[eval_idx])},
 "llm":{"e5":macro(yl,e5l,cl),"bge":macro(yl,bgel,cl),"old_selective":macro(yl,basel,cl),"meta":macro(yl,llm_blend,cl),"meta_gain":macro(yl,llm_blend,cl)-macro(yl,basel,cl)},
 "selection":{"candidate":selected_name,"alpha":alpha,"tune_oof":float(best.tune_oof),"positive_folds":int(best.positive_folds),"worst_fold":float(best.worst_fold),"teachers":top_llm}}
metrics["recommended"]=bool(metrics["manual"]["eval_gain"]>.0005 and metrics["llm"]["meta_gain"]>-.001)

prod_dir=MODELS/"production_meta";prod_dir.mkdir(exist_ok=True);fit_candidate(selected_cfg,np.arange(len(h)),[],2028,prod_dir)
for n in top_llm:shutil.copy2(MODELS/f"{n}.cbm",EXPORT/f"{n}.cbm");shutil.copy2(MODELS/f"{n}.json",EXPORT/f"{n}.json")
if (prod_dir/"meta_global.cbm").exists():shutil.copy2(prod_dir/"meta_global.cbm",EXPORT/"meta_global.cbm")
else:
    dst=EXPORT/"meta_categories";dst.mkdir(exist_ok=True)
    for p in prod_dir.glob("*.cbm"):shutil.copy2(p,dst/p.name)
grid.to_csv(EXPORT/"meta_search.csv",index=False);llm_table.to_csv(EXPORT/"llm_bank_metrics.csv",index=False)
percat=[]
for c in np.unique(ch[evaluation]):
    m=evaluation&(ch==c);percat.append({"category":c,"pairs":int(m.sum()),"old_selective":average_precision_score(yh[m],baseh[m]),"meta":average_precision_score(yh[m],honest[m]),"gain":average_precision_score(yh[m],honest[m])-average_precision_score(yh[m],baseh[m])})
pd.DataFrame(percat).sort_values("gain",ascending=False).to_csv(EXPORT/"manual_eval_by_category.csv",index=False)
(EXPORT/"metrics.json").write_text(json.dumps(metrics,ensure_ascii=False,indent=2));(EXPORT/"ensemble_config.json").write_text(json.dumps({"base":"E5 V5.2 + BGE top10 weight .20","meta_candidate":selected_cfg,"meta_alpha":alpha,"manual_features":MANUAL,"meta_features":FEATURES,"llm_teachers":top_llm},ensure_ascii=False,indent=2))
archive=shutil.make_archive("/kaggle/working/catboost_residual_bank_results","zip",EXPORT)
print("\nFINAL METRICS:\n",json.dumps(metrics,ensure_ascii=False,indent=2));print("\nVERDICT:","BUILD SUBMISSION" if metrics["recommended"] else "DO NOT SUBMIT");print("saved:",archive,"elapsed hours:",(time.time()-START)/3600)


INPUTS:
  human: /kaggle/input/datasets/mihailivanovvvv/hakaton-ozon-math-items/matches.parquet
  items_human: /kaggle/input/datasets/mihailivanovvvv/hakaton-ozon-math-items/items_human.parquet
  llm_full: /kaggle/input/datasets/mihailivanovvvv/hakaton-ozon-math-items/matches_llm.parquet
  items_full: /kaggle/input/datasets/mihailivanovvvv/hakaton-ozon-math-items/items.parquet
  polygon_2m: /kaggle/input/datasets/w4stdd/2m-parquet/llm_sample_2m.parquet
GPU: ['Tesla T4', 'Tesla T4']
E5 V5.2: /kaggle/input/models/danilzhukovv/ecup-product-matching-e5/pytorch/default/1 
BGE: /kaggle/input/models/danilzhukovv/ecup-product-matching-bge/pytorch/default/1
exact LLM holdout: 191,555, 0.9 min
LLM soft train: 2000000
profiles 2,588,017/2,588,017
human features 50,000/72,948 7463/s
human features 72,948/72,948 7374/s
LLM holdout features 50,000/191,555 7357/s
LLM holdout features 100,000/191,555 7354/s
LLM holdout features 150,000/191,555 7283/s
LLM holdout features 191,555/191,555 7272/s
LLM tra

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

E5 24/264,503 9/s
E5 24,024/264,503 128/s
E5 48,024/264,503 121/s
E5 72,024/264,503 116/s
E5 96,024/264,503 110/s
E5 120,024/264,503 105/s
E5 144,024/264,503 101/s
E5 168,024/264,503 98/s
E5 192,024/264,503 96/s
E5 216,024/264,503 94/s
E5 240,024/264,503 92/s
E5 264,024/264,503 90/s


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

BGE 512/264,503 303/s
BGE 20,992/264,503 139/s
BGE 41,472/264,503 113/s
BGE 61,952/264,503 98/s
BGE 82,432/264,503 86/s
BGE 102,912/264,503 78/s
BGE 123,392/264,503 73/s
BGE 143,872/264,503 70/s
BGE 164,352/264,503 67/s
BGE 184,832/264,503 65/s
BGE 205,312/264,503 64/s
BGE 225,792/264,503 62/s
BGE 246,272/264,503 61/s

BASELINES manual full: {'E5_V5.2': 0.7701344436902106, 'BGE': 0.7035273889698249, 'old_selective': 0.7714449371925742}
BASELINES exact LLM: {'E5_V5.2': 0.8166039584246516, 'BGE': 0.859740022180717, 'old_selective': 0.8216720423162187}
0:	learn: 0.6770780	test: 0.6770091	best: 0.6770091 (0)	total: 310ms	remaining: 20m 41s
250:	learn: 0.4504732	test: 0.4489084	best: 0.4489084 (250)	total: 20.9s	remaining: 5m 12s
500:	learn: 0.4441636	test: 0.4427697	best: 0.4427697 (500)	total: 41.2s	remaining: 4m 47s
750:	learn: 0.4409910	test: 0.4398545	best: 0.4398545 (750)	total: 1m 1s	remaining: 4m 25s
1000:	learn: 0.4387819	test: 0.4378724	best: 0.4378724 (1000)	total: 1m 21s	remaini

In [6]:
import gc,json,re,shutil,time,warnings
from difflib import SequenceMatcher
from pathlib import Path
import numpy as np,pandas as pd,polars as pl,pyarrow as pa,pyarrow.compute as pc,pyarrow.parquet as pq,torch
from catboost import CatBoostClassifier,Pool
from sklearn.metrics import average_precision_score
from transformers import AutoModelForSequenceClassification,AutoTokenizer
warnings.filterwarnings("ignore");T0=time.time();ROOT=Path("/kaggle/input");OUT=Path("/kaggle/working/bge_human_ft_meta");CACHE=OUT/"cache";OUT.mkdir(exist_ok=True);CACHE.mkdir(exist_ok=True)

def locate(name,prefer="hakaton-ozon-math-items"):
    z=list(ROOT.rglob(name));z.sort(key=lambda p:(prefer not in str(p),len(str(p))));assert z,f"Не найден {name}";return z[0]
HUMAN,LLM,IH,IF=locate("matches.parquet"),locate("matches_llm.parquet"),locate("items_human.parquet"),locate("items.parquet")
BANK=Path("/kaggle/input/datasets/danilzhukovv/ecup-catboost-residual-bank");assert BANK.exists(),BANK
def find_bge():
    z=[]
    for p in ROOT.rglob("config.json"):
        try:
            q=json.loads(p.read_text());w=p.parent/"model.safetensors"
            if q.get("hidden_size")==1024 and q.get("num_hidden_layers")==24 and w.exists():z.append(("human" in str(p).lower() or "ft" in str(p).lower(),w.stat().st_size,p.parent))
        except:pass
    assert z,"Не найдена BGE Human-FT";return sorted(z,reverse=True)[0][2]
BD=find_bge();print("GPU:",[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]);print("BGE:",BD,"\nBANK:",BANK)

def components(a,b):
    p={}
    def f(x):
        p.setdefault(x,x)
        while p[x]!=x:p[x]=p[p[x]];x=p[x]
        return x
    for x,y in zip(a,b):
        x,y=f(x),f(y)
        if x!=y:p[y]=x
    return np.fromiter((f(x) for x in a),np.int64,len(a))
def gmask(c,frac,seed):
    g=np.unique(c);r=np.random.RandomState(seed);s=set(g[r.rand(len(g))<frac]);return np.fromiter((x in s for x in c),bool,len(c))

HP,LP=CACHE/"human.parquet",CACHE/"llm.parquet"
if not HP.exists():
    h=pl.read_parquet(HUMAN,columns=["id1","id2","target"]).to_pandas();c=components(h.id1.to_numpy(),h.id2.to_numpy());m=gmask(c,.20,42);h=h.loc[m].copy();h["component"]=c[m];h.to_parquet(HP,index=False);print("human",len(h))
if not LP.exists():
    l=pl.read_parquet(LLM,columns=["id1","id2","target"]).to_pandas();c=components(l.id1.to_numpy(),l.id2.to_numpy());m=gmask(c,.03,13);l=l.loc[m].copy();vc=c[m];clean=((l.target<=.2)|(l.target>=.8)).to_numpy();l=l.loc[clean].copy();l["target_soft"]=l.target;l["target"]=(l.target>=.5).astype(np.int8);l["component"]=vc[clean];l.to_parquet(LP,index=False);print("LLM holdout",len(l))
h,l=pd.read_parquet(HP),pd.read_parquet(LP);assert len(h)==72948 and len(l)==191555,(len(h),len(l))

SPACE=re.compile(r"\s+");MULT=re.compile(r"[×хХ*]");TOK=re.compile(r"[0-9a-zа-я]+",re.I);SIZE=re.compile(r"(?:размер|р-р|size)\s*[:=\-]?\s*([0-9]{2,3}(?:[./-][0-9]{1,3})?|xxxs|xxs|xs|s|m|l|xl|xxl|xxxl)",re.I)
UNIT=re.compile(r"(\d+(?:[.,]\d+)?)\s*(мл|ml|л|l|мг|mg|г|gr|g|кг|kg|мм|mm|см|cm|м|m|вт|w|квт|kw|шт|pcs|pack)\b",re.I);PACK=re.compile(r"(\d+)\s*[xх×*]\s*(\d+(?:[.,]\d+)?)\s*(мл|ml|л|l|мг|mg|г|gr|g|кг|kg|шт|pcs)",re.I)
ALIASES={"brand":["бренд","brand","производитель"],"article":["артикул","sku","oem","партномер","part number","partnumber","код производителя"],"model":["модель","model"],"size":["размер производителя","размер обуви","размер одежды","размер","size","рост"],"color":["основной цвет","цвет","color","расцветка"],"material":["материал верха","материал","material"],"gender":["пол","gender"]}
HOMO=str.maketrans({"а":"a","в":"b","с":"c","е":"e","н":"h","к":"k","м":"m","о":"o","р":"p","т":"t","х":"x","у":"y"});COL={x.replace("ё","е"):y for x,y in {"черный":"черный","чёрный":"черный","белый":"белый","серый":"серый","серебристый":"серебристый","красный":"красный","бордовый":"бордовый","синий":"синий","голубой":"голубой","зеленый":"зеленый","зелёный":"зеленый","желтый":"желтый","жёлтый":"желтый","оранжевый":"оранжевый","розовый":"розовый","фиолетовый":"фиолетовый","бежевый":"бежевый","коричневый":"коричневый","золотой":"золотой","золотистый":"золотой"}.items()}
def norm(x):return "" if x is None else SPACE.sub(" ",MULT.sub("x",str(x).lower().replace("ё","е").replace(",","."))).strip()
def adict(x):
    if isinstance(x,dict):d=x
    elif isinstance(x,str):
        try:d=json.loads(x)
        except:d={}
    else:d={}
    return {norm(k):norm(v) for k,v in d.items() if norm(k) and norm(v)} if isinstance(d,dict) else {}
def pick(d,a):
    for z in a:
        for k,v in d.items():
            if z in k:return v
    return ""
def compact(x,n=96):return SPACE.sub(" ",re.sub(r"[^0-9a-zа-я.+/_\- ]+"," ",norm(x))).strip()[:n]
def code(x):return re.sub(r"[^0-9a-z]+","",compact(x).translate(HOMO))
def snorm(x):return re.sub(r"\s*/\s*","/",re.sub(r"\s*-\s*","-",compact(x,64).replace("–","-").replace("—","-")))
def color(x):
    t=compact(x,64);f=[v for k,v in COL.items() if k in t];return "/".join(sorted(set(f))) if f else t
def qtokens(text):
    def cv(v,u):
        u=u.lower()
        if u in {"мл","ml"}:return "volume_ml",v
        if u in {"л","l"}:return "volume_ml",v*1000
        if u in {"мг","mg"}:return "mass_g",v/1000
        if u in {"г","gr","g"}:return "mass_g",v
        if u in {"кг","kg"}:return "mass_g",v*1000
        if u in {"мм","mm"}:return "length_mm",v
        if u in {"см","cm"}:return "length_mm",v*10
        if u in {"м","m"}:return "length_mm",v*1000
        if u in {"вт","w"}:return "power_w",v
        if u in {"квт","kw"}:return "power_w",v*1000
        return "count",v
    t=norm(text);o=set()
    for c,a,u in PACK.findall(t):k,v=cv(float(a.replace(",",".")),u);o|={f"{k}:{round(float(c)*v,4)}",f"pack_count:{round(float(c),4)}"}
    for a,u in UNIT.findall(t):k,v=cv(float(a.replace(",",".")),u);o.add(f"{k}:{round(v,4)}")
    return frozenset(o)
class Item:
    __slots__=("name","tokens","brand","article","model","size","color","material","gender","quantity","category")
    def __init__(self,n,a,c):
        d=adict(a);self.name=norm(n);self.tokens=frozenset(x for x in TOK.findall(self.name) if len(x)>=2);self.brand=compact(pick(d,ALIASES["brand"]));self.article=code(pick(d,ALIASES["article"]));self.model=code(pick(d,ALIASES["model"]));m=SIZE.search(self.name);self.size=snorm(pick(d,ALIASES["size"])) or (snorm(m.group(1)) if m else "");self.color=color(pick(d,ALIASES["color"])) or "/".join(sorted(set(v for k,v in COL.items() if k in self.name)));self.material=compact(pick(d,ALIASES["material"]));self.gender=compact(pick(d,ALIASES["gender"]));self.category=str(c);self.quantity=qtokens(self.name+" "+" ".join(d.values())[:1800])
def jac(a,b):return len(a&b)/len(a|b) if a|b else 0.
def con(a,b):return len(a&b)/max(1,min(len(a),len(b))) if a and b else 0.
def eq(a,b):return -1. if not a or not b else float(a==b)
def structured(a,b):
    x,y=a.name,b.name;v=[float(bool(x) and x==y),float(bool(x) and bool(y) and (x in y or y in x)),SequenceMatcher(None,x,y).ratio(),jac(a.tokens,b.tokens),con(a.tokens,b.tokens),min(len(x),len(y))/max(1,max(len(x),len(y))),min(abs(len(x)-len(y))/100,5)]
    for p,q in [(a.brand,b.brand),(a.article,b.article),(a.model,b.model),(a.size,b.size),(a.color,b.color),(a.material,b.material),(a.gender,b.gender)]:v += [eq(p,q),float(bool(p) and bool(q))]
    v += [jac(a.quantity,b.quantity),float(bool(a.quantity) and bool(b.quantity) and not a.quantity&b.quantity),float(bool(a.quantity) and bool(b.quantity)),SequenceMatcher(None,a.article,b.article).ratio() if a.article and b.article else 0.,SequenceMatcher(None,a.model,b.model).ratio() if a.model and b.model else 0.];return np.asarray(v,np.float32)

BKEY=["бренд","артикул","партномер","oem","код","модель","размер","цвет","объем","обьем","вес","тип","материал","количество"];C2L=str.maketrans("аеорсухАЕОРСУХКМТВНЗЅІі","aeopcyxAEOPCYXKMTBH3SIi");L2C=str.maketrans("aeopcyxAEOPCYX","аеорсухАЕОРСУХ");BU=re.compile(r"(\d+(?:[.,]\d+)?)\s*(мл|ml|л|l|мг|mg|г|g|гр|кг|kg|мм|mm|см|cm|м|мб|mb|гб|gb|тб|tb|вт|w|квт|kw|мач|mah)\b",re.I);BM={"мл":("ml",1),"ml":("ml",1),"л":("ml",1000),"l":("ml",1000),"мг":("g",.001),"mg":("g",.001),"г":("g",1),"g":("g",1),"гр":("g",1),"кг":("g",1000),"kg":("g",1000),"мм":("mm",1),"mm":("mm",1),"см":("mm",10),"cm":("mm",10),"м":("mm",1000),"мб":("gb",.001),"mb":("gb",.001),"гб":("gb",1),"gb":("gb",1),"тб":("gb",1024),"tb":("gb",1024),"вт":("w",1),"w":("w",1),"квт":("w",1000),"kw":("w",1000),"мач":("mah",1),"mah":("mah",1)};QR=[re.compile(r"(\d+)\s*шт"),re.compile(r"набор\w*\s+из\s+(\d+)"),re.compile(r"(\d+)\s*(?:набор|упаков|комплект)\w*\s+по\s+(\d+)"),re.compile(r"[xх*](\d+)\b")]
def btext(n,a):
    p=[str(n) if n is not None else ""]
    try:d=json.loads(a) if isinstance(a,str) else {}
    except:d={}
    if isinstance(d,dict) and d:
        low={str(k).lower():str(v) for k,v in d.items() if v};q=[];used=set()
        for w in BKEY:
            for k,v in low.items():
                if w in k and k not in used:q.append(f"{k}:{v}");used.add(k)
        p.append(" ; ".join(q+[f"{k}:{v}" for k,v in low.items() if k not in used])[:520])
    s=" | ".join(p).replace("ё","е").replace("Ё","Е");z=[]
    for t in s.split():c=sum("\u0400"<=x<="\u04ff" for x in t);ll=sum(x.isascii() and x.isalpha() for x in t);z.append(t.translate(C2L if ll>=c else L2C) if c and ll else t)
    s=re.sub(r"[×хХ](?=\d)","x"," ".join(z));u=set()
    for m in BU.finditer(s):k,v=BM[m.group(2).lower()];u.add(f"{float(m.group(1).replace(',','.'))*v:g}{k}")
    ex=["ед: "+" ".join(sorted(u)[:12])] if u else [];q=None;m=QR[2].search(s.lower())
    if m:q=int(m.group(1))*int(m.group(2))
    else:
        for r in (QR[0],QR[1],QR[3]):
            m=r.search(s.lower())
            if m:q=int(m.group(1));break
    if q and 1<q<=1000:ex.append(f"кол-во: {q}")
    return (s+(" | "+" | ".join(ex) if ex else ""))[:2000]

HF,LF,HC,LC,TX=CACHE/"human_features.npy",CACHE/"llm_features.npy",CACHE/"human_cat.npy",CACHE/"llm_cat.npy",CACHE/"texts.parquet"
if not all(x.exists() for x in [HF,LF,HC,LC,TX]):
    need=set(h.id1)|set(h.id2)|set(l.id1)|set(l.id2);P={};rows=[];rem=set(need);t=time.time()
    for ip in [IF,IH]:
        if not rem:break
        pf=pq.ParquetFile(ip);vals=pa.array(list(rem),type=pf.schema_arrow.field("id").type)
        for ba in pf.iter_batches(columns=["id","name","attributes","category"],batch_size=300000,use_threads=True):
            ids=ba.column(ba.schema.get_field_index("id"));sel=ba.filter(pc.is_in(ids,value_set=vals))
            for i,n,a,c in sel.to_pandas().itertuples(index=False,name=None):
                if i in rem:P[i]=Item(n,a,c);rows.append((i,btext(n,a)));rem.remove(i)
        print("profiles",len(P),"/",len(need))
    assert not rem,f"Missing items {len(rem)}";pd.DataFrame(rows,columns=["id","text"]).to_parquet(TX,index=False)
    def build(d,fp,cp):
        x=np.lib.format.open_memmap(fp,"w+",dtype=np.float32,shape=(len(d),26));c=np.empty(len(d),dtype="<U64");a=d.id1.to_numpy();b=d.id2.to_numpy()
        for s in range(0,len(d),25000):
            e=min(s+25000,len(d))
            for j in range(s,e):x[j]=structured(P[a[j]],P[b[j]]);c[j]=P[a[j]].category
            print(fp.name,e,"/",len(d))
        x.flush();np.save(cp,c)
    build(h,HF,HC);build(l,LF,LC);del P,rows;gc.collect();print("preprocess min",(time.time()-t)/60)
Xh,Xl=np.load(HF,mmap_mode="r"),np.load(LF,mmap_mode="r");ch,cl=np.load(HC),np.load(LC);yh,yl=h.target.to_numpy(np.int8),l.target.to_numpy(np.int8);MAN=[f"structured_{i:02d}" for i in range(26)]

def rank(x):return pd.Series(np.asarray(x)).rank(method="average",pct=True).to_numpy()
def crank(x,c):
    o=np.empty(len(x),np.float64)
    for z in np.unique(c):m=c==z;o[m]=rank(np.asarray(x)[m])
    return o
def macro(y,p,c):return float(np.mean([average_precision_score(y[c==z],p[c==z]) for z in np.unique(c) if np.unique(y[c==z]).size>1]))
def blend(a,b,c,w):return (1-w)*crank(a,c)+w*crank(b,c)

BH,BL=CACHE/"bge_hft_h.npy",CACHE/"bge_hft_l.npy"
if BH.exists():bh,bl=np.load(BH),np.load(BL)
else:
    tm=dict(pd.read_parquet(TX).itertuples(index=False,name=None));p=pd.concat([h[["id1","id2"]],l[["id1","id2"]]],ignore_index=True);sp=len(h);a=p.id1.to_numpy();b=p.id2.to_numpy();order=np.argsort([len(tm[x])+len(tm[y]) for x,y in zip(a,b)],kind="stable");pred=np.empty(len(p),np.float32);tok=AutoTokenizer.from_pretrained(BD,local_files_only=True);net=AutoModelForSequenceClassification.from_pretrained(BD,local_files_only=True,dtype=torch.float32).cuda().eval()
    if torch.cuda.device_count()>1:net=torch.nn.DataParallel(net,device_ids=list(range(torch.cuda.device_count())))
    bs=256;t=time.time()
    with torch.inference_mode():
        for s in range(0,len(order),bs):
            ix=order[s:s+bs];e=tok([tm[a[i]] for i in ix],[tm[b[i]] for i in ix],padding=True,truncation=True,max_length=320,return_tensors="pt");e={k:v.cuda(non_blocking=True) for k,v in e.items()}
            with torch.autocast("cuda",dtype=torch.float16):pred[ix]=torch.sigmoid(net(**e).logits.squeeze(-1).float()).cpu().numpy()
            if s//bs%80==0:print(f"BGE-HFT {s+len(ix):,}/{len(order):,} {(s+len(ix))/max(1,time.time()-t):.0f}/s")
    bh,bl=pred[:sp],pred[sp:];np.save(BH,bh);np.save(BL,bl);del net,tok,tm,p,pred;gc.collect();torch.cuda.empty_cache()

def frame(x,c):
    d=pd.DataFrame(np.asarray(x),columns=MAN,copy=False);d.insert(0,"category",np.asarray(c).astype(str));return d
Fh,Fl=frame(Xh,ch),frame(Xl,cl);TEACH=["llm_d9_s44","llm_d10_s88","llm_d8_s33"];th={};tl={}
for n in TEACH:
    m=CatBoostClassifier();z=list(BANK.rglob(n+".cbm"));assert z,n;m.load_model(z[0]);th[n]=m.predict_proba(Fh)[:,1];tl[n]=m.predict_proba(Fl)[:,1]
def meta(x,c,b,t):
    d=frame(x,c);d["bge_hft"]=b;d["bge_logit"]=np.log(np.clip(b,1e-5,1-1e-5)/np.clip(1-b,1e-5,1));d["bge_rank"]=crank(b,c);d["bge_conf"]=abs(b-.5)
    for n,p in t.items():d["teacher_"+n]=p
    a=np.column_stack(list(t.values()));d["teacher_mean"]=a.mean(1);d["teacher_std"]=a.std(1);d["teacher_min"]=a.min(1);d["teacher_max"]=a.max(1);return d
Mh,Ml=meta(Xh,ch,bh,th),meta(Xl,cl,bl,tl);br=crank(bh,ch);tune=gmask(h.component.to_numpy(),.5,2026);ev=~tune;g=np.unique(h.loc[tune,"component"]);r=np.random.RandomState(2027);fm={x:i%5 for i,x in enumerate(r.permutation(g))};fold=np.full(len(h),-1,np.int8);fold[tune]=[fm[x] for x in h.loc[tune,"component"]];cnt=pd.Series(ch).value_counts();med=cnt.median();cw={k:float(np.clip((med/v)**.5,.7,1.5)) for k,v in cnt.items()};CFG=[("d6",6,1400,.025,8),("d7",7,1600,.022,12),("d8",8,1600,.018,18)]
def weight(ix):
    w=np.asarray([cw[x] for x in ch[ix]],np.float32);y=yh[ix];q=br[ix];return w*(1+3*((y==0)&(q>.8))+1.5*((y==1)&(q<.2)))
def fit(cfg,tr,outs,seed,path=None):
    n,d,it,lr,l2=cfg;m=CatBoostClassifier(iterations=it,depth=d,learning_rate=lr,l2_leaf_reg=l2,loss_function="Logloss",random_strength=1,task_type="GPU",devices="0:1" if torch.cuda.device_count()>1 else "0",border_count=128,random_seed=seed,allow_writing_files=False,verbose=False);m.fit(Pool(Mh.iloc[tr],yh[tr],cat_features=["category"],weight=weight(tr)));p=[m.predict_proba(x)[:,1] for x in outs]
    if path:m.save_model(path)
    del m;gc.collect();torch.cuda.empty_cache();return p
rows=[];store={}
for cfg in CFG:
    o=np.full(len(h),np.nan,np.float32)
    for f in range(5):tr=np.flatnonzero(tune&(fold!=f));va=np.flatnonzero(tune&(fold==f));o[va]=fit(cfg,tr,[Mh.iloc[va]],100*cfg[1]+f)[0];print(cfg[0],"fold",f,"done")
    store[cfg[0]]=o
    for a in np.arange(0,.251,.025):
        gains=[]
        for f in range(5):m=tune&(fold==f);gains.append(macro(yh[m],blend(bh[m],o[m],ch[m],a),ch[m])-macro(yh[m],bh[m],ch[m]))
        p=blend(bh[tune],o[tune],ch[tune],a);rows.append({"candidate":cfg[0],"alpha":float(a),"tune":macro(yh[tune],p,ch[tune]),"gain":macro(yh[tune],p,ch[tune])-macro(yh[tune],bh[tune],ch[tune]),"positive_folds":sum(x>0 for x in gains),"worst_fold":min(gains)})
grid=pd.DataFrame(rows).sort_values(["tune","positive_folds"],ascending=False);stable=grid[(grid.positive_folds>=4)&(grid.gain>=.0003)&(grid.worst_fold>=-.001)];best=stable.iloc[0] if len(stable) else grid.iloc[0];cfg=next(x for x in CFG if x[0]==best.candidate);alpha=float(best.alpha);print("\nTOP\n",grid.head(15).to_string(index=False));print("SELECTED",cfg,"alpha",alpha)
ti,ei=np.flatnonzero(tune),np.flatnonzero(ev);ep,lp=fit(cfg,ti,[Mh.iloc[ei],Ml],999,OUT/"meta_eval.cbm");eb,lb=blend(bh[ei],ep,ch[ei],alpha),blend(bl,lp,cl,alpha);metrics={"public_lb_anchor":.5432564386950458,"warning":"Human-FT мог видеть human labels","manual":{"bge_full":macro(yh,bh,ch),"bge_eval":macro(yh[ei],bh[ei],ch[ei]),"meta_eval":macro(yh[ei],eb,ch[ei]),"gain":macro(yh[ei],eb,ch[ei])-macro(yh[ei],bh[ei],ch[ei])},"llm":{"bge":macro(yl,bl,cl),"meta":macro(yl,lb,cl),"gain":macro(yl,lb,cl)-macro(yl,bl,cl)},"selection":{"candidate":cfg[0],"alpha":alpha,"tune_gain":float(best.gain),"positive_folds":int(best.positive_folds),"worst_fold":float(best.worst_fold)}}
EXP=OUT/"export";EXP.mkdir(exist_ok=True);fit(cfg,np.arange(len(h)),[],2028,EXP/"meta_production.cbm");grid.to_csv(EXP/"search.csv",index=False);(EXP/"metrics.json").write_text(json.dumps(metrics,ensure_ascii=False,indent=2));(EXP/"config.json").write_text(json.dumps({"teachers":TEACH,"candidate":cfg,"alpha":alpha},ensure_ascii=False,indent=2))
for n in TEACH:shutil.copy2(list(BANK.rglob(n+".cbm"))[0],EXP/(n+".cbm"))
z=shutil.make_archive("/kaggle/working/bge_human_ft_meta_results","zip",EXP);print("\nFINAL\n",json.dumps(metrics,ensure_ascii=False,indent=2));print("saved",z,"hours",(time.time()-T0)/3600)


GPU: ['Tesla T4', 'Tesla T4']
BGE: /kaggle/input/models/danilzhukovv/ecup-bge-human-ft/pytorch/default/1 
BANK: /kaggle/input/datasets/danilzhukovv/ecup-catboost-residual-bank
human 72948
LLM holdout 191555
profiles 394813 / 394813
human_features.npy 25000 / 72948
human_features.npy 50000 / 72948
human_features.npy 72948 / 72948
llm_features.npy 25000 / 191555
llm_features.npy 50000 / 191555
llm_features.npy 75000 / 191555
llm_features.npy 100000 / 191555
llm_features.npy 125000 / 191555
llm_features.npy 150000 / 191555
llm_features.npy 175000 / 191555
llm_features.npy 191555 / 191555
preprocess min 4.507709602514903


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

BGE-HFT 256/264,503 146/s
BGE-HFT 20,736/264,503 327/s
BGE-HFT 41,216/264,503 267/s
BGE-HFT 61,696/264,503 235/s
BGE-HFT 82,176/264,503 211/s
BGE-HFT 102,656/264,503 192/s
BGE-HFT 123,136/264,503 179/s
BGE-HFT 143,616/264,503 171/s
BGE-HFT 164,096/264,503 165/s
BGE-HFT 184,576/264,503 160/s
BGE-HFT 205,056/264,503 156/s
BGE-HFT 225,536/264,503 153/s
BGE-HFT 246,016/264,503 151/s
d6 fold 0 done
d6 fold 1 done
d6 fold 2 done
d6 fold 3 done
d6 fold 4 done
d7 fold 0 done
d7 fold 1 done
d7 fold 2 done
d7 fold 3 done
d7 fold 4 done
d8 fold 0 done
d8 fold 1 done
d8 fold 2 done
d8 fold 3 done
d8 fold 4 done

TOP
 candidate  alpha     tune      gain  positive_folds  worst_fold
       d6  0.000 0.823016  0.000000               0    0.000000
       d7  0.000 0.823016  0.000000               0    0.000000
       d8  0.000 0.823016  0.000000               0    0.000000
       d8  0.025 0.822945 -0.000071               4   -0.000079
       d6  0.025 0.822928 -0.000088               4   -0.000024
   